In [1]:
# CELLULE 1 - Setup environnement LOCAL (CPU) - Semaine 1 : collecte, nettoyage, anonymisation

import subprocess, sys, platform

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

print("Machine :", platform.platform())
print("Python  :", sys.version)

# Librairies nécessaires UNIQUEMENT pour la phase de données (pas d'entraînement ici)
packages = [
    "datasets",          # chargement corpus HuggingFace (MediQA, MedQuAD, etc.)
    "huggingface_hub",   # accès + upload vers HF Hub
    "presidio-analyzer",  # détection d'entités sensibles (RGPD)
    "presidio-anonymizer",# masquage des entités sensibles
    "spacy",             # NER (moteur utilisé par Presidio)
    "pandas",            # manipulation tabulaire
    "scikit-learn",      # split train/val/test stratifié
]

run(f"{sys.executable} -m pip install -q " + " ".join(packages))

# Modèle spaCy français pour la détection d'entités nommées (noms, prénoms, lieux...)
run(f"{sys.executable} -m spacy download fr_core_news_md")
# Modèle spaCy anglais (car corpus bilingue FR/EN)
run(f"{sys.executable} -m spacy download en_core_web_md")

print("\n✅ Setup terminé (mode CPU - pas de torch/transformers pour l'instant).")
import datasets, spacy, pandas as pd
print("datasets:", datasets.__version__)
print("spacy:", spacy.__version__)
print("pandas:", pd.__version__)

Machine : macOS-10.16-x86_64-i386-64bit
Python  : 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]
$ /opt/anaconda3/envs/triage-poc/bin/python -m pip install -q datasets huggingface_hub presidio-analyzer presidio-anonymizer spacy pandas scikit-learn
$ /opt/anaconda3/envs/triage-poc/bin/python -m spacy download fr_core_news_md
     ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/45.8 MB 115.7 kB/s eta 0:05:02


  Resuming download fr_core_news_md-3.8.0-py3-none-any.whl (11.0 MB/45.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 1.1 MB/s  0:00:30m0:00:0100:02m
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
$ /opt/anaconda3/envs/triage-poc/bin/python -m spacy download en_core_web_md
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 1.1 MB/s  0:00:30m0:00:0100:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')

✅ Setup terminé (mode CPU - pas de torch/transformers pour l'instant).
datasets: 5.0.1
spacy: 3.8.15
pandas: 3.0.5


In [4]:
# CELLULE 2 (corrigée) - Inventaire et chargement des corpus médicaux (FR/EN)

import subprocess, sys
subprocess.run(f"{sys.executable} -m pip install -q ipywidgets", shell=True)

from datasets import load_dataset
import pandas as pd

# IDs HuggingFace corrigés
sources = {
    "FrenchMedMCQA": "nthngdy/frenchmedmcqa",
    "MediQAl": "ANR-MALADES/MediQAl",
    "MedQuAD": "keivalya/MedQuad-MedicalQnADataset",
    "UltraMedical-Preference (DPO)": "TsinghuaC3I/UltraMedical-Preference",
}

loaded_datasets = {}
errors = {}

for name, hf_id in sources.items():
    print(f"\n{'='*60}\nTentative de chargement : {name} ({hf_id})\n{'='*60}")
    try:
        ds = load_dataset(hf_id)
        loaded_datasets[name] = ds
        print(f"✅ Succès. Splits disponibles : {list(ds.keys())}")
        for split in ds.keys():
            print(f"   - {split}: {len(ds[split])} exemples")
        first_split = list(ds.keys())[0]
        print(f"   Colonnes : {ds[first_split].column_names}")
        print(f"   Exemple  : {ds[first_split][0]}")
    except Exception as e:
        errors[name] = str(e)
        print(f"❌ Échec : {e}")

print(f"\n\n{'='*60}\nRÉSUMÉ\n{'='*60}")
print(f"Chargés avec succès : {list(loaded_datasets.keys())}")
print(f"Échecs : {list(errors.keys())}")


Tentative de chargement : FrenchMedMCQA (nthngdy/frenchmedmcqa)
✅ Succès. Splits disponibles : ['train', 'validation', 'test']
   - train: 595 exemples
   - validation: 164 exemples
   - test: 321 exemples
   Colonnes : ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers']
   Exemple  : {'id': '230bac49b0fe863b772410bc8d01a025f63c3c999065480131d6334abd2efeff', 'question': 'Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha', 'answer_a': "Sont formées de noyaux d'hélium", 'answer_b': 'Sont peu pénétrantes', 'answer_c': "Toute l'énergie qu'elles transportent est cédée au long d'un parcours de quelques centimètres dans l'air", 'answer_d': 'Sont arrêtées par une feuille de papier', 'answer_e': 'Sont peu ionisantes', 'correct_answers': 4, 'number_correct_answers': 0}

Tentative de chargement : MediQAl (ANR-MALADES/MediQAl)
❌ Échec : Config name is missing.
Please pick one amon

In [5]:
# CELLULE 2bis - Chargement de MediQAl avec ses 3 configurations

from datasets import load_dataset

mediqal_configs = ["mcqu", "mcqm", "oeq"]  # mcqu=QCM unique, mcqm=QCM multiple, oeq=question ouverte (open-ended)

mediqal_datasets = {}

for config in mediqal_configs:
    print(f"\n{'='*60}\nMediQAl - config : {config}\n{'='*60}")
    try:
        ds = load_dataset("ANR-MALADES/MediQAl", config)
        mediqal_datasets[config] = ds
        print(f"✅ Succès. Splits : {list(ds.keys())}")
        for split in ds.keys():
            print(f"   - {split}: {len(ds[split])} exemples")
        first_split = list(ds.keys())[0]
        print(f"   Colonnes : {ds[first_split].column_names}")
        print(f"   Exemple  : {ds[first_split][0]}")
    except Exception as e:
        print(f"❌ Échec : {e}")


MediQAl - config : mcqu
✅ Succès. Splits : ['train', 'validation', 'test']
   - train: 10113 exemples
   - validation: 2561 exemples
   - test: 4343 exemples
   Colonnes : ['id', 'clinical_case', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'task', 'medical_subject', 'question_type']
   Exemple  : {'id': '7543', 'clinical_case': "Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladi

In [6]:
# CELLULE 3 - Exploration des sujets médicaux (pour cibler les cas "urgence/triage")

import pandas as pd

# Convertir en DataFrame pour explorer facilement
df_oeq = mediqal_datasets["oeq"]["test"].to_pandas()
df_mcqu = mediqal_datasets["mcqu"]["train"].to_pandas()
df_mcqm = mediqal_datasets["mcqm"]["train"].to_pandas()

print("="*60)
print("Distribution des medical_subject - MediQAl OEQ (cas cliniques ouverts)")
print("="*60)
print(df_oeq["medical_subject"].value_counts())

print("\n" + "="*60)
print("Distribution des question_type - MediQAl OEQ")
print("="*60)
print(df_oeq["question_type"].value_counts())

# Filtrer spécifiquement sur Emergency Medicine
df_oeq_urgences = df_oeq[df_oeq["medical_subject"] == "Emergency Medicine"]
print(f"\n✅ Nombre de cas 'Emergency Medicine' dans OEQ : {len(df_oeq_urgences)}")

print("\n" + "="*60)
print("Distribution des medical_subject - MediQAl MCQU")
print("="*60)
print(df_mcqu["medical_subject"].value_counts())

# Aperçu de quelques exemples "Emergency Medicine" du OEQ
print("\n" + "="*60)
print("Aperçu de 3 cas Emergency Medicine (OEQ)")
print("="*60)
for i, row in df_oeq_urgences.head(3).iterrows():
    print(f"\n--- Cas clinique {row['id']} ---")
    print(row['clinical_case'][:500])
    print(f"\nQuestion : {row['question']}")
    print(f"Réponse  : {row['answer'][:300]}")

Distribution des medical_subject - MediQAl OEQ (cas cliniques ouverts)
medical_subject
Pharmacy                            1544
Pediatric Cardiology                1225
Cardiology                           217
Nephro-Urology                       193
Pulmonology                          184
Intensive Care                       154
Neurology                            150
Gynecology and Obstetrics            149
Otorhinolaryngology (ENT)            126
Hematology                           124
Emergency Medicine                   109
Orthopedics                           93
Infectious Diseases                   88
Forensic Medicine and Toxicology      86
Psychiatry                            80
Hepato-Gastroenterology               68
Genetics                              65
Dermatology                           54
Pediatrics                            47
Rheumatology                          37
Ophthalmology                         31
Endocrinology and Metabolism          24
Parasitolog

In [7]:
import os
os.makedirs("data_raw", exist_ok=True)

In [8]:
# CELLULE 4 - Constitution du vivier de cas cliniques pertinents pour l'annotation priorité

import pandas as pd

# Spécialités où la notion de gravité/urgence est la plus pertinente
specialites_urgence = [
    "Emergency Medicine",
    "Intensive Care",
    "Cardiology",
    "Pulmonology",
    "Pediatric Cardiology",
]

df_pool = df_oeq[df_oeq["medical_subject"].isin(specialites_urgence)].copy()
print(f"Total lignes (questions) dans ces spécialités : {len(df_pool)}")

# Dédupliquer par cas clinique unique
cas_uniques = df_pool.drop_duplicates(subset=["clinical_case"])
print(f"Nombre de CAS CLINIQUES UNIQUES : {len(cas_uniques)}")

print("\nRépartition par spécialité (cas uniques) :")
print(cas_uniques["medical_subject"].value_counts())

# Vérifier combien de ces cas mentionnent des constantes vitales explicites
# (signal fort pour permettre une annotation objective de la priorité)
mots_cles_constantes = ["FC", "PA", "FR", "SpO2", "SaO2", "GCS", "Glasgow", "T°", "T ="]

def a_constantes(texte):
    return any(mot in texte for mot in mots_cles_constantes)

cas_uniques["a_constantes_vitales"] = cas_uniques["clinical_case"].apply(a_constantes)
print(f"\nCas avec constantes vitales explicites : {cas_uniques['a_constantes_vitales'].sum()} / {len(cas_uniques)}")

# Sauvegarde du vivier pour l'étape d'annotation (cellule 5)
cas_uniques_avec_constantes = cas_uniques[cas_uniques["a_constantes_vitales"]].reset_index(drop=True)
print(f"\n✅ Vivier final pour annotation manuelle : {len(cas_uniques_avec_constantes)} cas cliniques uniques")

cas_uniques_avec_constantes.to_csv("data_raw/vivier_annotation_priorite.csv", index=False)
print("Sauvegardé dans data_raw/vivier_annotation_priorite.csv")

Total lignes (questions) dans ces spécialités : 1889
Nombre de CAS CLINIQUES UNIQUES : 483

Répartition par spécialité (cas uniques) :
medical_subject
Pediatric Cardiology    363
Pulmonology              35
Intensive Care           34
Emergency Medicine       26
Cardiology               25
Name: count, dtype: int64


TypeError: argument of type 'float' is not iterable

In [ ]:
# CELLULE 4 (corrigée) - Constitution du vivier de cas cliniques pour l'annotation priorité

import pandas as pd

specialites_urgence = [
    "Emergency Medicine",
    "Intensive Care",
    "Cardiology",
    "Pulmonology",
    "Pediatric Cardiology",
]

df_pool = df_oeq[df_oeq["medical_subject"].isin(specialites_urgence)].copy()

# Nettoyage : retirer les cas cliniques manquants/vides avant tout traitement
avant = len(df_pool)
df_pool = df_pool.dropna(subset=["clinical_case"])
df_pool = df_pool[df_pool["clinical_case"].astype(str).str.strip() != ""]
print(f"Lignes retirées (clinical_case manquant/vide) : {avant - len(df_pool)}")
print(f"Total lignes (questions) dans ces spécialités après nettoyage : {len(df_pool)}")

# Dédupliquer par cas clinique unique
cas_uniques = df_pool.drop_duplicates(subset=["clinical_case"]).copy()
print(f"Nombre de CAS CLINIQUES UNIQUES : {len(cas_uniques)}")

print("\nRépartition par spécialité (cas uniques) :")
print(cas_uniques["medical_subject"].value_counts())

# Détection des constantes vitales explicites (fonction robuste au typage)
mots_cles_constantes = ["FC", "PA", "FR", "SpO2", "SaO2", "GCS", "Glasgow", "T°", "T ="]

def a_constantes(texte):
    texte = str(texte)  # sécurise le typage, quel que soit l'input
    return any(mot in texte for mot in mots_cles_constantes)

cas_uniques["a_constantes_vitales"] = cas_uniques["clinical_case"].map(a_constantes)
print(f"\nCas avec constantes vitales explicites : {cas_uniques['a_constantes_vitales'].sum()} / {len(cas_uniques)}")

# Sauvegarde du vivier pour l'étape d'annotation (cellule 5)
import os
os.makedirs("data_raw", exist_ok=True)

cas_uniques_avec_constantes = cas_uniques[cas_uniques["a_constantes_vitales"]].reset_index(drop=True)
print(f"\n✅ Vivier final pour annotation manuelle : {len(cas_uniques_avec_constantes)} cas cliniques uniques")

cas_uniques_avec_constantes.to_csv("data_raw/vivier_annotation_priorite.csv", index=False)
print("Sauvegardé dans data_raw/vivier_annotation_priorite.csv")

In [9]:
# CELLULE 4bis - Inspection du format Pediatric Cardiology + élargissement du vivier

import re

# 1. Inspecter 3 cas Pediatric Cardiology pour comprendre pourquoi ils ne matchent pas
df_pedcard = cas_uniques[cas_uniques["medical_subject"] == "Pediatric Cardiology"]
print("Exemples de cas Pediatric Cardiology (pour voir le format des constantes) :\n")
for i, row in df_pedcard.head(3).iterrows():
    print(f"--- Cas {row['id']} ---")
    print(row['clinical_case'][:400])
    print()

# 2. Élargir la recherche à TOUT le dataset OEQ (pas seulement 5 spécialités)
#    avec une regex plus robuste (espaces, accents, variantes) + mots-clés de gravité clinique
df_all = df_oeq.dropna(subset=["clinical_case"]).copy()
df_all = df_all[df_all["clinical_case"].astype(str).str.strip() != ""]
cas_uniques_all = df_all.drop_duplicates(subset=["clinical_case"]).copy()
print(f"\nTotal cas cliniques uniques (tout OEQ) : {len(cas_uniques_all)}")

# Regex robuste pour constantes vitales (tolère espaces, casse, accents simples)
pattern_constantes = re.compile(
    r"(FC\s*[:=]|FR\s*[:=]|SpO2|Sp02|SaO2|PA\s*[:=]|GCS|Glasgow|"
    r"satur\w*|tension\s+art|fréquence\s+card|fréquence\s+respi|"
    r"T°|T\s*[:=]\s*\d|détresse|choc\s|marbrures|cyanose|coma)",
    re.IGNORECASE
)

def a_signal_gravite(texte):
    return bool(pattern_constantes.search(str(texte)))

cas_uniques_all["a_signal_gravite"] = cas_uniques_all["clinical_case"].map(a_signal_gravite)

print(f"\nCas avec signal de gravité (regex élargie, tout OEQ) : {cas_uniques_all['a_signal_gravite'].sum()} / {len(cas_uniques_all)}")

print("\nRépartition par spécialité des cas retenus :")
print(cas_uniques_all[cas_uniques_all["a_signal_gravite"]]["medical_subject"].value_counts())

# Sauvegarde du nouveau vivier élargi
vivier_elargi = cas_uniques_all[cas_uniques_all["a_signal_gravite"]].reset_index(drop=True)
vivier_elargi.to_csv("data_raw/vivier_annotation_priorite_v2.csv", index=False)
print(f"\n✅ Nouveau vivier : {len(vivier_elargi)} cas cliniques uniques -> data_raw/vivier_annotation_priorite_v2.csv")

Exemples de cas Pediatric Cardiology (pour voir le format des constantes) :

--- Cas 343 ---


TypeError: 'float' object is not subscriptable

In [10]:
# CELLULE 5 - Sanity check Pharmacy + définition de la grille d'annotation priorité

# 1. Vérifier les faux positifs potentiels dans "Pharmacy"
df_pharmacy_check = vivier_elargi[vivier_elargi["medical_subject"] == "Pharmacy"]
print("Aperçu de 3 cas 'Pharmacy' retenus (à vérifier) :\n")
for i, row in df_pharmacy_check.head(3).iterrows():
    print(f"--- Cas {row['id']} ---")
    print(row['clinical_case'][:300])
    print()

# 2. Définition formelle de la grille d'annotation (à documenter dans le rapport)
GRILLE_ANNOTATION = """
GRILLE D'ANNOTATION - NIVEAU DE PRIORITÉ DE TRIAGE
=====================================================
Inspirée des classifications de triage FRENCH (France) et Manchester (international),
adaptée en 3 niveaux pour ce POC.

Annotateur : IA Engineer junior (non-professionnel de santé) 
--> Limite méthodologique à documenter explicitement dans le rapport.

1. URGENCE MAXIMALE (rouge / vital engagé immédiatement)
   Critères (au moins UN présent) :
   - SpO2 < 90% en air ambiant
   - FR < 8 ou > 30 /min
   - FC < 40 ou > 140 /min (adulte) - seuils différents en pédiatrie
   - PA systolique < 90 mmHg ou imprenable
   - Trouble de conscience (GCS < 13, coma, incohérence)
   - Signes de choc (marbrures, cyanose généralisée, pouls filants)
   - Détresse respiratoire ou circulatoire explicite

2. URGENCE MODÉRÉE (orange/jaune - prise en charge rapide mais pas immédiate)
   Critères :
   - Constantes vitales limites mais stables
   - Douleur intense (EVA > 6)
   - Pathologie potentiellement grave mais patient stable actuellement
   - Nécessite surveillance rapprochée et examens complémentaires urgents

3. DIFFÉRÉE (vert - peut attendre)
   Critères :
   - Constantes vitales normales
   - Absence de signe de gravité
   - Pathologie chronique stable ou bénigne
   - Pas de risque vital ou fonctionnel à court terme

NB : En cas de doute entre deux niveaux -> toujours choisir le niveau le PLUS URGENT 
(principe de précaution en triage réel).
"""
print(GRILLE_ANNOTATION)

NameError: name 'vivier_elargi' is not defined

In [11]:
# CELLULE 6 (v2) - Annotation en mode texte simple (fonctionne partout, sans widgets)

import pandas as pd
import os

VIVIER_PATH = "data_raw/vivier_annotation_priorite_v2.csv"
ANNOTATIONS_PATH = "data_annotated/annotations_priorite.csv"

os.makedirs("data_annotated", exist_ok=True)

vivier = pd.read_csv(VIVIER_PATH)

if os.path.exists(ANNOTATIONS_PATH):
    annotations = pd.read_csv(ANNOTATIONS_PATH)
else:
    annotations = pd.DataFrame(columns=["id", "priorite", "commentaire"])

deja_annotes = set(annotations["id"].astype(str))
restant = vivier[~vivier["id"].astype(str).isin(deja_annotes)].reset_index(drop=True)

print(f"Total : {len(vivier)} | Déjà annoté : {len(deja_annotes)} | Restant : {len(restant)}")
print("\nPour chaque cas, tape :")
print("  1 = 🔴 Urgence maximale")
print("  2 = 🟠 Urgence modérée")
print("  3 = 🟢 Différée")
print("  s = passer (skip, incertain)")
print("  q = arrêter et sauvegarder\n")

mapping = {"1": "urgence_maximale", "2": "urgence_moderee", "3": "differee"}
nouvelles_annotations = []

for i, row in restant.iterrows():
    print("="*80)
    print(f"Cas {i+1}/{len(restant)}  (id={row['id']} — {row['medical_subject']})")
    print("-"*80)
    print(row["clinical_case"])
    print(f"\nQuestion : {row.get('question', '')}")
    print("-"*80)

    reponse = input("Ton choix (1/2/3/s/q) : ").strip().lower()

    if reponse == "q":
        print("Arrêt demandé.")
        break
    if reponse == "s":
        continue
    if reponse not in mapping:
        print("⚠️ Réponse non reconnue, cas passé automatiquement.")
        continue

    commentaire = input("Commentaire (optionnel, Entrée pour passer) : ").strip()

    nouvelles_annotations.append({
        "id": row["id"],
        "priorite": mapping[reponse],
        "commentaire": commentaire
    })

    # Sauvegarde toutes les 10 annotations
    if len(nouvelles_annotations) % 10 == 0:
        df_temp = pd.concat([annotations, pd.DataFrame(nouvelles_annotations)], ignore_index=True)
        df_temp.to_csv(ANNOTATIONS_PATH, index=False)
        print(f"💾 Sauvegardé ({len(df_temp)} au total)")

# Sauvegarde finale
if nouvelles_annotations:
    annotations = pd.concat([annotations, pd.DataFrame(nouvelles_annotations)], ignore_index=True)
    annotations.to_csv(ANNOTATIONS_PATH, index=False)

print(f"\n✅ Session terminée. Total annoté cumulé : {len(annotations)} / {len(vivier)}")

Total : 355 | Déjà annoté : 214 | Restant : 201

Pour chaque cas, tape :
  1 = 🔴 Urgence maximale
  2 = 🟠 Urgence modérée
  3 = 🟢 Différée
  s = passer (skip, incertain)
  q = arrêter et sauvegarder

Cas 1/201  (id=1326 — Pediatric Cardiology)
--------------------------------------------------------------------------------
Après un diagnostic anténatal d’hypoplasie du cœur gauche, vous accueillez à 10 minutes de vie un nouveau-né cyanosé à 80%, modérément polypnéique chez qui vous palpez tous les pouls. Vous confirmer le diagnostic de la cardiopathie en échographie (atrésie mitrale et aortique). Deux heures après son arrivée, la situation clinique se dégrade avec une détresse respiratoire sévère. La radiographie montre un œdème pulmonaire bilatéral. Il n’y a pas de dysfonctionnement du ventricule droit ni de fuite tricuspide ; le canal artériel est largement ouvert et fonction de façon adéquate.

Question : Quelle explication fournissez-vous à cette évolution ? Comment pouvez-vous la t

Ton choix (1/2/3/s/q) :  q


Arrêt demandé.

✅ Session terminée. Total annoté cumulé : 214 / 355


In [12]:
# CELLULE 7 - Vérification de la distribution des annotations

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")

print(f"Total annoté : {len(annotations)}\n")

print("Distribution des priorités :")
distribution = annotations["priorite"].value_counts()
print(distribution)

print("\nEn pourcentage :")
print((distribution / len(annotations) * 100).round(1).astype(str) + " %")

# Aperçu des commentaires laissés (utile pour repérer les cas où tu as hésité)
avec_commentaire = annotations[annotations["commentaire"].notna() & (annotations["commentaire"] != "")]
print(f"\nCas avec commentaire (doute/note) : {len(avec_commentaire)}")
if len(avec_commentaire) > 0:
    print(avec_commentaire[["id", "priorite", "commentaire"]].head(10))

Total annoté : 214

Distribution des priorités :
priorite
urgence_maximale    101
urgence_moderee      80
differee             33
Name: count, dtype: int64

En pourcentage :
priorite
urgence_maximale    47.2 %
urgence_moderee     37.4 %
differee            15.4 %
Name: count, dtype: str

Cas avec commentaire (doute/note) : 38
       id          priorite commentaire
5      80          differee         3.0
6      85          differee         3.0
7     109   urgence_moderee         1.0
17    264  urgence_maximale         1.0
136  1227  urgence_maximale           1
152  1321  urgence_maximale           q
154    48   urgence_moderee           3
155  4890          differee           2
156   749  urgence_maximale           3
157  4855          differee           2


In [23]:
# CELLULE 8 - Relecture des cas classés "urgence_maximale" pour recalibrage

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")
vivier = pd.read_csv("data_raw/vivier_annotation_priorite_v2.csv")

# On ne relit QUE les cas actuellement en urgence_maximale
a_relire = annotations[annotations["priorite"] == "urgence_maximale"].copy()
print(f"Cas à relire : {len(a_relire)}")
print("\nCritère recalibré : réserve '1' aux situations d'instabilité AIGUË, ACTUELLE,")
print("potentiellement vitale DANS L'HEURE. Si le cas décrit une pathologie sérieuse")
print("mais stabilisée/suivie/programmée -> reclasse en 2 (modérée).\n")
print("Tape Entrée seul pour GARDER en urgence_maximale (pas de changement).\n")

mapping = {"1": "urgence_maximale", "2": "urgence_moderee", "3": "differee"}
changements = 0

for idx, row in a_relire.iterrows():
    case_row = vivier[vivier["id"] == row["id"]]
    if case_row.empty:
        continue
    case_row = case_row.iloc[0]

    print("="*80)
    print(f"id={row['id']} — {case_row['medical_subject']} — actuellement : {row['priorite']}")
    print("-"*80)
    print(case_row["clinical_case"][:600])
    print(f"\nQuestion : {case_row.get('question', '')}")
    print("-"*80)

    reponse = input("Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) : ").strip().lower()

    if reponse == "q":
        print("Relecture arrêtée.")
        break
    if reponse in mapping:
        nouvelle_priorite = mapping[reponse]
        if nouvelle_priorite != row["priorite"]:
            annotations.loc[annotations["id"] == row["id"], "priorite"] = nouvelle_priorite
            changements += 1
            print(f"✅ Reclassé en {nouvelle_priorite}")
    # sinon (Entrée vide) -> on garde tel quel

annotations.to_csv("data_annotated/annotations_priorite.csv", index=False)
print(f"\n💾 Sauvegardé. {changements} cas reclassés sur cette session.")
print("\nNouvelle distribution :")
print(annotations["priorite"].value_counts())

Cas à relire : 147

Critère recalibré : réserve '1' aux situations d'instabilité AIGUË, ACTUELLE,
potentiellement vitale DANS L'HEURE. Si le cas décrit une pathologie sérieuse
mais stabilisée/suivie/programmée -> reclasse en 2 (modérée).

Tape Entrée seul pour GARDER en urgence_maximale (pas de changement).

id=1 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.

 Examen d’entrée : 
 Constantes :
 - FC = 130bpm
 - PA = 130/80 aux deux bras
 - FR = 40/min
 - SpO2 = 80%
 - T = 38 ,5°C
 Tirage sus-sternal
 Cyanose
 Sueurs
 Auscultation : 
 - Ronchi bilatéraux et sibilants
 - Tympanisme bilatéral
 Hépatomégalie douloureuse
 Reflux hépato-jugulaire
 Turgescence jugulaire
 Hippocratisme digital
 Cyanose unguéale
 
 ATCD 
 Fumeur (80 PA)
 BPCO
 
 Traitement habituel
 β2 mimétiques 
 
 HDM : depuis 1 semaine : ne peut plus fumer, expectora

Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  


id=7 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
M. Shoot, 30 ans.
 Agitation à la sortie d’une boîte de nuit, amené par les pompiers.
 Son amie rapporte qu’il aime « bien s’amuser » en prenant de l’héroïne ou de la cocaïne.
 Voici ses paramètres vitaux :
 - PA 180/110
 - FC 130
 - T 39.5
 - FR 25
 - SpO2 96%. Quelques heures     après son arrivée, M. Shoot présente des vomissements incoercibles puis un épisode de convulsions. Son amie, inquiète et étudiante en médecine, vous demande quelles sont vos hypothèses diagnostiques.

Question : Que lui répondez-vous ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  


id=9 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Mme Barbie, 72 ans
 Percutée par un bus alors qu’elle était à Vélib’
 Pas de casque, pour ne pas abîmer son brushing
 A son arrivée aux urgences :
 - Trauma crânien
 - Plaie au coir chevelu
 - Impotence fonctionnelle coude gauche. Alors que vous suturez sa paie, Mme Barbie se plaint de céphalées, cervicalgies, présente une somnolence.
 Paramètres vitaux :
 GSC = 12
 TA = 90/52
 FC = 95
 T = 37,5°C
 SpO2 = 98%
 FR = 21

Question : Que recherchez-vous à l’examen neurologique ?
--------------------------------------------------------------------------------


...

id=1212 — Pediatric Cardiology — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Un nouveau-né de 3 jours de vie vous est confié pour une cyanose réfractaire à l’oxygène. Il n’a pas de détresse circulatoire ou respiratoire. Vous auscultez un souffle systolique 3/6 et un B2 fort unique. L’ECG est normal. La radiographie montre une silhouette en œuf avec un médiastin étroit et des poumons normalement perfusés. L’échographie montre une D-transposition des gros vaisseaux avec CIV et sténose pulmonaire. Quelques heures plus tard, la cyanose devient profonde sans détresse respiratoire. La radiographie de thorax montre des poumons très clairs.

Question : Que faites-vous ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  1


id=1214 — Pediatric Cardiology — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Un nouveau-né de 3 jours de vie vous est confié pour une cyanose réfractaire à l’oxygène. Il n’a pas de détresse circulatoire ou respiratoire. Vous auscultez un souffle systolique 3/6 et un B2 fort unique. L’ECG est normal. La radiographie montre une silhouette en œuf avec un médiastin étroit et des poumons normalement perfusés. L’échographie montre une D-transposition des gros vaisseaux avec CIV et sténose pulmonaire. Quelques heures plus tard, la cyanose devient profonde sans détresse respiratoire. La radiographie de thorax montre des poumons très clairs. Un an plus tard, l’enfant va bien. L

Question : Quel traitement proposez-vous ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  q


Relecture arrêtée.

💾 Sauvegardé. 30 cas reclassés sur cette session.

Nouvelle distribution :
priorite
urgence_maximale    117
urgence_moderee      35
differee              2
Name: count, dtype: int64


In [13]:
# CELLULE 7 - Vérification de la distribution des annotations

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")

print(f"Total annoté : {len(annotations)}\n")

print("Distribution des priorités :")
distribution = annotations["priorite"].value_counts()
print(distribution)

print("\nEn pourcentage :")
print((distribution / len(annotations) * 100).round(1).astype(str) + " %")

# Aperçu des commentaires laissés (utile pour repérer les cas où tu as hésité)
avec_commentaire = annotations[annotations["commentaire"].notna() & (annotations["commentaire"] != "")]
print(f"\nCas avec commentaire (doute/note) : {len(avec_commentaire)}")
if len(avec_commentaire) > 0:
    print(avec_commentaire[["id", "priorite", "commentaire"]].head(10))

Total annoté : 214

Distribution des priorités :
priorite
urgence_maximale    101
urgence_moderee      80
differee             33
Name: count, dtype: int64

En pourcentage :
priorite
urgence_maximale    47.2 %
urgence_moderee     37.4 %
differee            15.4 %
Name: count, dtype: str

Cas avec commentaire (doute/note) : 38
       id          priorite commentaire
5      80          differee         3.0
6      85          differee         3.0
7     109   urgence_moderee         1.0
17    264  urgence_maximale         1.0
136  1227  urgence_maximale           1
152  1321  urgence_maximale           q
154    48   urgence_moderee           3
155  4890          differee           2
156   749  urgence_maximale           3
157  4855          differee           2


In [25]:
# CELLULE 8 - Relecture des cas classés "urgence_maximale" pour recalibrage

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")
vivier = pd.read_csv("data_raw/vivier_annotation_priorite_v2.csv")

# On ne relit QUE les cas actuellement en urgence_maximale
a_relire = annotations[annotations["priorite"] == "urgence_maximale"].copy()
print(f"Cas à relire : {len(a_relire)}")
print("\nCritère recalibré : réserve '1' aux situations d'instabilité AIGUË, ACTUELLE,")
print("potentiellement vitale DANS L'HEURE. Si le cas décrit une pathologie sérieuse")
print("mais stabilisée/suivie/programmée -> reclasse en 2 (modérée).\n")
print("Tape Entrée seul pour GARDER en urgence_maximale (pas de changement).\n")

mapping = {"1": "urgence_maximale", "2": "urgence_moderee", "3": "differee"}
changements = 0

for idx, row in a_relire.iterrows():
    case_row = vivier[vivier["id"] == row["id"]]
    if case_row.empty:
        continue
    case_row = case_row.iloc[0]

    print("="*80)
    print(f"id={row['id']} — {case_row['medical_subject']} — actuellement : {row['priorite']}")
    print("-"*80)
    print(case_row["clinical_case"][:600])
    print(f"\nQuestion : {case_row.get('question', '')}")
    print("-"*80)

    reponse = input("Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) : ").strip().lower()

    if reponse == "q":
        print("Relecture arrêtée.")
        break
    if reponse in mapping:
        nouvelle_priorite = mapping[reponse]
        if nouvelle_priorite != row["priorite"]:
            annotations.loc[annotations["id"] == row["id"], "priorite"] = nouvelle_priorite
            changements += 1
            print(f"✅ Reclassé en {nouvelle_priorite}")
    # sinon (Entrée vide) -> on garde tel quel

annotations.to_csv("data_annotated/annotations_priorite.csv", index=False)
print(f"\n💾 Sauvegardé. {changements} cas reclassés sur cette session.")
print("\nNouvelle distribution :")
print(annotations["priorite"].value_counts())

Cas à relire : 117

Critère recalibré : réserve '1' aux situations d'instabilité AIGUË, ACTUELLE,
potentiellement vitale DANS L'HEURE. Si le cas décrit une pathologie sérieuse
mais stabilisée/suivie/programmée -> reclasse en 2 (modérée).

Tape Entrée seul pour GARDER en urgence_maximale (pas de changement).

id=1 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.

 Examen d’entrée : 
 Constantes :
 - FC = 130bpm
 - PA = 130/80 aux deux bras
 - FR = 40/min
 - SpO2 = 80%
 - T = 38 ,5°C
 Tirage sus-sternal
 Cyanose
 Sueurs
 Auscultation : 
 - Ronchi bilatéraux et sibilants
 - Tympanisme bilatéral
 Hépatomégalie douloureuse
 Reflux hépato-jugulaire
 Turgescence jugulaire
 Hippocratisme digital
 Cyanose unguéale
 
 ATCD 
 Fumeur (80 PA)
 BPCO
 
 Traitement habituel
 β2 mimétiques 
 
 HDM : depuis 1 semaine : ne peut plus fumer, expectora

Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  2


✅ Reclassé en urgence_moderee
id=7 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
M. Shoot, 30 ans.
 Agitation à la sortie d’une boîte de nuit, amené par les pompiers.
 Son amie rapporte qu’il aime « bien s’amuser » en prenant de l’héroïne ou de la cocaïne.
 Voici ses paramètres vitaux :
 - PA 180/110
 - FC 130
 - T 39.5
 - FR 25
 - SpO2 96%. Quelques heures     après son arrivée, M. Shoot présente des vomissements incoercibles puis un épisode de convulsions. Son amie, inquiète et étudiante en médecine, vous demande quelles sont vos hypothèses diagnostiques.

Question : Que lui répondez-vous ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  2


✅ Reclassé en urgence_moderee
id=9 — Emergency Medicine — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Mme Barbie, 72 ans
 Percutée par un bus alors qu’elle était à Vélib’
 Pas de casque, pour ne pas abîmer son brushing
 A son arrivée aux urgences :
 - Trauma crânien
 - Plaie au coir chevelu
 - Impotence fonctionnelle coude gauche. Alors que vous suturez sa paie, Mme Barbie se plaint de céphalées, cervicalgies, présente une somnolence.
 Paramètres vitaux :
 GSC = 12
 TA = 90/52
 FC = 95
 T = 37,5°C
 SpO2 = 98%
 FR = 21

Question : Que recherchez-vous à l’examen neurologique ?
--------------------------------------------------------------------------------


...

id=673 — Pediatric Cardiology — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Un nouveau-né de mère lupique naît à terme après une grossesse normale en bloc auriculo-ventriculaire complet. Sa fréquence cardiaque est à 60/mn.

Question : Faut-il mettre une sonde d’entraînement électrosystolique transitoire ou un stimulateur cardiaque d’emblée ? Pourquoi ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  1


id=701 — Pediatric Cardiology — actuellement : urgence_maximale
--------------------------------------------------------------------------------
Un nouveau-né est admis pour une atrésie pulmonaire à septum interventriculaire intact. Le ventricule droit est bipartite sans chambre trabéculée mais la valve tricuspide a une taille correcte. Cette condition d'absence de fistules coronaro-cardiaque est remplie et une perforation-dilatation de la valve pulmonaire est faite. Malgré tout, vous ne pouvez arrêter les PGE1. Quelques mois plus tard, la cyanose s’est majorée et vous n’enregistrez dans l’artère pulmonaire qu’un flux systolique. Il persiste un shunt droite-gauche par un foramen ovale. Le ventricule droit a grandi en proportion du vent

Question : Que proposez-vous ?
--------------------------------------------------------------------------------


Nouveau choix (1/2/3, Entrée = garder, q = arrêter la relecture) :  q


Relecture arrêtée.

💾 Sauvegardé. 28 cas reclassés sur cette session.

Nouvelle distribution :
priorite
urgence_maximale    89
urgence_moderee     63
differee             2
Name: count, dtype: int64


In [26]:
# CELLULE 7 - Vérification de la distribution des annotations

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")

print(f"Total annoté : {len(annotations)}\n")

print("Distribution des priorités :")
distribution = annotations["priorite"].value_counts()
print(distribution)

print("\nEn pourcentage :")
print((distribution / len(annotations) * 100).round(1).astype(str) + " %")

# Aperçu des commentaires laissés (utile pour repérer les cas où tu as hésité)
avec_commentaire = annotations[annotations["commentaire"].notna() & (annotations["commentaire"] != "")]
print(f"\nCas avec commentaire (doute/note) : {len(avec_commentaire)}")
if len(avec_commentaire) > 0:
    print(avec_commentaire[["id", "priorite", "commentaire"]].head(10))

Total annoté : 154

Distribution des priorités :
priorite
urgence_maximale    89
urgence_moderee     63
differee             2
Name: count, dtype: int64

En pourcentage :
priorite
urgence_maximale    57.8 %
urgence_moderee     40.9 %
differee             1.3 %
Name: count, dtype: str

Cas avec commentaire (doute/note) : 6
       id          priorite commentaire
5      80          differee         3.0
6      85          differee         3.0
7     109   urgence_moderee         1.0
17    264  urgence_maximale         1.0
136  1227  urgence_maximale           1
152  1321  urgence_maximale           q


In [27]:
# CELLULE 9 - Recherche de cas cliniques SANS signal de gravité (pour équilibrer "différée")

import re
import pandas as pd

# Reprendre tout OEQ, nettoyé
df_all_clean = df_oeq.dropna(subset=["clinical_case"]).copy()
df_all_clean = df_all_clean[df_all_clean["clinical_case"].astype(str).str.strip() != ""]
cas_uniques_tous = df_all_clean.drop_duplicates(subset=["clinical_case"]).copy()

# Exclure les cas déjà présents dans notre vivier initial (déjà traités, annotés ou non)
ids_deja_vus = set(vivier_elargi["id"].astype(str))
cas_restants = cas_uniques_tous[~cas_uniques_tous["id"].astype(str).isin(ids_deja_vus)].copy()
print(f"Cas non encore examinés : {len(cas_restants)}")

# Regex de gravité (la même que Cellule 4bis) pour EXCLURE les cas qui la matchent
pattern_gravite = re.compile(
    r"(FC\s*[:=]|FR\s*[:=]|SpO2|Sp02|SaO2|PA\s*[:=]|GCS|Glasgow|"
    r"satur\w*|tension\s+art|fréquence\s+card|fréquence\s+respi|"
    r"T°|T\s*[:=]\s*\d|détresse|choc\s|marbrures|cyanose|coma|"
    r"urgence|réanimation|intubé|hypotension|hypoxém)",
    re.IGNORECASE
)

def sans_gravite(texte):
    return not bool(pattern_gravite.search(str(texte)))

cas_restants["sans_signal_gravite"] = cas_restants["clinical_case"].map(sans_gravite)

pool_differee = cas_restants[cas_restants["sans_signal_gravite"]].copy()
print(f"Cas candidats 'différée' (sans signal de gravité détecté) : {len(pool_differee)}")

print("\nRépartition par spécialité :")
print(pool_differee["medical_subject"].value_counts().head(15))

# On tire un échantillon aléatoire de 60 cas pour l'annotation complémentaire (reproductible)
echantillon_differee = pool_differee.sample(n=min(60, len(pool_differee)), random_state=42).reset_index(drop=True)

echantillon_differee.to_csv("data_raw/vivier_differee_complement.csv", index=False)
print(f"\n✅ Échantillon de {len(echantillon_differee)} cas sauvegardé -> data_raw/vivier_differee_complement.csv")

# Aperçu de 2 exemples pour vérifier la pertinence
print("\nAperçu :")
for i, row in echantillon_differee.head(2).iterrows():
    print(f"\n--- {row['medical_subject']} ---")
    print(row['clinical_case'][:300])

Cas non encore examinés : 656
Cas candidats 'différée' (sans signal de gravité détecté) : 533

Répartition par spécialité :
medical_subject
Pharmacy                     185
Pediatric Cardiology         182
Gynecology and Obstetrics     45
Neurology                     22
Otorhinolaryngology (ENT)     19
Pulmonology                   12
Cardiology                     7
Psychiatry                     6
Genetics                       5
Hematology                     5
Dermatology                    5
Emergency Medicine             4
Rheumatology                   4
Orthopedics                    4
Nephro-Urology                 4
Name: count, dtype: int64

✅ Échantillon de 60 cas sauvegardé -> data_raw/vivier_differee_complement.csv

Aperçu :

--- Otorhinolaryngology (ENT) ---
Un enfant de deux ans et demi est adressé pour des otites moyennes aigües récidivant toutes les 6 semaines environ. Cet enfant a déjà eu un bilan allergologique et une recherche de carence martiale, négatifs. A l’ex

In [1]:
# CELLULE 10 - Annotation du complément "différée" (60 cas sans signal de gravité détecté)

import pandas as pd
import os

VIVIER_PATH = "data_raw/vivier_differee_complement.csv"
ANNOTATIONS_PATH = "data_annotated/annotations_priorite.csv"  # même fichier, on cumule

vivier_complement = pd.read_csv(VIVIER_PATH)
annotations = pd.read_csv(ANNOTATIONS_PATH)

deja_annotes = set(annotations["id"].astype(str))
restant = vivier_complement[~vivier_complement["id"].astype(str).isin(deja_annotes)].reset_index(drop=True)

print(f"Total complément : {len(vivier_complement)} | Déjà annoté : {len(vivier_complement) - len(restant)} | Restant : {len(restant)}")
print("\nRappel : ces cas ont été présélectionnés SANS signal de gravité détecté.")
print("Attends-toi à majoritairement voir des cas 'différée', mais reste vigilant :")
print("le filtre automatique peut rater un vrai cas grave sans mot-clé explicite.\n")
print("  1 = 🔴 Urgence maximale   2 = 🟠 Urgence modérée   3 = 🟢 Différée")
print("  s = passer   q = arrêter et sauvegarder\n")

mapping = {"1": "urgence_maximale", "2": "urgence_moderee", "3": "differee"}
nouvelles_annotations = []

for i, row in restant.iterrows():
    print("="*80)
    print(f"Cas {i+1}/{len(restant)}  (id={row['id']} — {row['medical_subject']})")
    print("-"*80)
    print(row["clinical_case"])
    print(f"\nQuestion : {row.get('question', '')}")
    print("-"*80)

    reponse = input("Ton choix (1/2/3/s/q) : ").strip().lower()

    if reponse == "q":
        print("Arrêt demandé.")
        break
    if reponse == "s":
        continue
    if reponse not in mapping:
        print("⚠️ Réponse non reconnue, cas passé automatiquement.")
        continue

    commentaire = input("Commentaire (optionnel, Entrée pour passer) : ").strip()

    nouvelles_annotations.append({
        "id": row["id"],
        "priorite": mapping[reponse],
        "commentaire": commentaire
    })

    if len(nouvelles_annotations) % 10 == 0:
        df_temp = pd.concat([annotations, pd.DataFrame(nouvelles_annotations)], ignore_index=True)
        df_temp.to_csv(ANNOTATIONS_PATH, index=False)
        print(f"💾 Sauvegardé ({len(df_temp)} au total)")

if nouvelles_annotations:
    annotations = pd.concat([annotations, pd.DataFrame(nouvelles_annotations)], ignore_index=True)
    annotations.to_csv(ANNOTATIONS_PATH, index=False)

print(f"\n✅ Session terminée. Total annoté cumulé : {len(annotations)}")

Total complément : 60 | Déjà annoté : 58 | Restant : 2

Rappel : ces cas ont été présélectionnés SANS signal de gravité détecté.
Attends-toi à majoritairement voir des cas 'différée', mais reste vigilant :
le filtre automatique peut rater un vrai cas grave sans mot-clé explicite.

  1 = 🔴 Urgence maximale   2 = 🟠 Urgence modérée   3 = 🟢 Différée
  s = passer   q = arrêter et sauvegarder

Cas 1/2  (id=533 — Pediatric Cardiology)
--------------------------------------------------------------------------------
Vous faites une scintigraphie myocardique au Thallium 201 chez un enfant de 12 ans ayant eu une correction anatomique de transposition des gros vaisseaux en période néonatale. Celle-ci montre un défaut de perfusion scintigraphique réversible dans le territoire antéro-septo-apical.

Question : Quelle est la signification de ce résultat?
--------------------------------------------------------------------------------


Ton choix (1/2/3/s/q) :  2
Commentaire (optionnel, Entrée pour passer) :  


Cas 2/2  (id=1208 — Pediatric Cardiology)
--------------------------------------------------------------------------------
Un enfant de 12 ans a eu une réparation complète d’atrésie pulmonaire avec CIV à l’aide d’un tube valvé de 14 mm à l’âge de 3 ans. Il va bien mais la pression dans le ventricule droit est à 125 mm Hg pour une pression aortique à 100 mm Hg.

Question : Quel traitement proposez-vous si les branches pulmonaires sont normales ainsi que la pression pulmonaire au delà du tube ?
--------------------------------------------------------------------------------


Ton choix (1/2/3/s/q) :  1
Commentaire (optionnel, Entrée pour passer) :  



✅ Session terminée. Total annoté cumulé : 214


In [2]:
# CELLULE 11 - Bilan complet de l'annotation

import pandas as pd

annotations = pd.read_csv("data_annotated/annotations_priorite.csv")

print(f"Total annoté : {len(annotations)}\n")

distribution = annotations["priorite"].value_counts()
print("Distribution des priorités :")
print(distribution)

print("\nEn pourcentage :")
print((distribution / len(annotations) * 100).round(1).astype(str) + " %")

# Vérifier s'il reste des valeurs suspectes dans priorite (issues des doubles frappes "11", "12" etc.)
valeurs_valides = {"urgence_maximale", "urgence_moderee", "differee"}
valeurs_invalides = annotations[~annotations["priorite"].isin(valeurs_valides)]
if len(valeurs_invalides) > 0:
    print(f"\n⚠️ {len(valeurs_invalides)} lignes avec une valeur de priorité invalide (à corriger) :")
    print(valeurs_invalides[["id", "priorite"]])
else:
    print("\n✅ Aucune valeur invalide détectée dans la colonne priorite.")

# Vérifier les doublons d'id (au cas où)
doublons = annotations[annotations.duplicated(subset=["id"], keep=False)]
print(f"\nDoublons d'id détectés : {len(doublons)}")
if len(doublons) > 0:
    print(doublons.sort_values("id"))

Total annoté : 214

Distribution des priorités :
priorite
urgence_maximale    101
urgence_moderee      80
differee             33
Name: count, dtype: int64

En pourcentage :
priorite
urgence_maximale    47.2 %
urgence_moderee     37.4 %
differee            15.4 %
Name: count, dtype: str

✅ Aucune valeur invalide détectée dans la colonne priorite.

Doublons d'id détectés : 0


In [14]:
# CELLULE 12 - Schéma de métadonnées unifié pour le dataset SFT

import pandas as pd
import uuid

# ============================================================
# SCHÉMA CIBLE (documenté pour le rapport)
# ============================================================
# id                : identifiant unique généré
# langue            : "fr" ou "en"
# source            : nom du corpus d'origine
# source_id         : id original dans le corpus source
# instruction       : la question/consigne posée au modèle
# contexte_patient  : cas clinique / contexte (peut être vide si non applicable)
# reponse           : la réponse attendue (réponse cible pour le SFT)
# specialite        : domaine médical (medical_subject / qtype / etc.)
# priorite          : urgence_maximale / urgence_moderee / differee / None (rempli seulement pour le sous-ensemble annoté)
# niveau_confiance  : "annote_manuellement" / "source_originale" (traçabilité qualité)
# split             : train / val / test (à assigner ensuite)
# ============================================================

SCHEMA_COLONNES = [
    "id", "langue", "source", "source_id", "instruction",
    "contexte_patient", "reponse", "specialite", "priorite",
    "niveau_confiance", "split"
]

def nouvelle_ligne(langue, source, source_id, instruction, contexte_patient,
                    reponse, specialite, priorite=None, niveau_confiance="source_originale"):
    return {
        "id": str(uuid.uuid4())[:8],
        "langue": langue,
        "source": source,
        "source_id": str(source_id),
        "instruction": str(instruction).strip() if instruction else "",
        "contexte_patient": str(contexte_patient).strip() if pd.notna(contexte_patient) else "",
        "reponse": str(reponse).strip() if reponse else "",
        "specialite": str(specialite) if pd.notna(specialite) else "",
        "priorite": priorite,
        "niveau_confiance": niveau_confiance,
        "split": None,  # assigné plus tard (Cellule split train/val/test)
    }

print("✅ Schéma défini :", SCHEMA_COLONNES)
print("\nExemple de ligne vide :")
print(nouvelle_ligne("fr", "test", "1", "Question test", "Contexte test", "Réponse test", "Cardiology"))

✅ Schéma défini : ['id', 'langue', 'source', 'source_id', 'instruction', 'contexte_patient', 'reponse', 'specialite', 'priorite', 'niveau_confiance', 'split']

Exemple de ligne vide :
{'id': '37fc5b36', 'langue': 'fr', 'source': 'test', 'source_id': '1', 'instruction': 'Question test', 'contexte_patient': 'Contexte test', 'reponse': 'Réponse test', 'specialite': 'Cardiology', 'priorite': None, 'niveau_confiance': 'source_originale', 'split': None}


In [15]:
# CELLULE 12bis - Rechargement des variables nécessaires (après redémarrage du kernel)

import pandas as pd
import uuid
from datasets import load_dataset

# Recharger MediQAl oeq
mediqal_oeq = load_dataset("ANR-MALADES/MediQAl", "oeq")
df_oeq = mediqal_oeq["test"].to_pandas()
print(f"✅ df_oeq rechargé : {len(df_oeq)} lignes")

# Redéfinir le schéma et la fonction nouvelle_ligne (identiques à la Cellule 12)
SCHEMA_COLONNES = [
    "id", "langue", "source", "source_id", "instruction",
    "contexte_patient", "reponse", "specialite", "priorite",
    "niveau_confiance", "split"
]

def nouvelle_ligne(langue, source, source_id, instruction, contexte_patient,
                    reponse, specialite, priorite=None, niveau_confiance="source_originale"):
    return {
        "id": str(uuid.uuid4())[:8],
        "langue": langue,
        "source": source,
        "source_id": str(source_id),
        "instruction": str(instruction).strip() if instruction else "",
        "contexte_patient": str(contexte_patient).strip() if pd.notna(contexte_patient) else "",
        "reponse": str(reponse).strip() if reponse else "",
        "specialite": str(specialite) if pd.notna(specialite) else "",
        "priorite": priorite,
        "niveau_confiance": niveau_confiance,
        "split": None,
    }

print("✅ Schéma et fonction nouvelle_ligne redéfinis")

✅ df_oeq rechargé : 4969 lignes
✅ Schéma et fonction nouvelle_ligne redéfinis


In [16]:
# CELLULE 13 - Conversion de MediQAl OEQ vers le schéma unifié

import pandas as pd

# Charger nos annotations de priorité
annotations = pd.read_csv("data_annotated/annotations_priorite.csv")
annotations["id"] = annotations["id"].astype(str)
map_priorite = dict(zip(annotations["id"], annotations["priorite"]))

print(f"Nombre de cas avec priorité annotée disponible : {len(map_priorite)}")

lignes_mediqal = []

for _, row in df_oeq.dropna(subset=["clinical_case", "question", "answer"]).iterrows():
    source_id = str(row["id"])
    priorite = map_priorite.get(source_id, None)  # None si pas dans notre sous-ensemble annoté
    niveau_confiance = "annote_manuellement" if priorite else "source_originale"

    lignes_mediqal.append(nouvelle_ligne(
        langue="fr",
        source="MediQAl_oeq",
        source_id=source_id,
        instruction=row["question"],
        contexte_patient=row["clinical_case"],
        reponse=row["answer"],
        specialite=row["medical_subject"],
        priorite=priorite,
        niveau_confiance=niveau_confiance
    ))

df_mediqal_final = pd.DataFrame(lignes_mediqal, columns=SCHEMA_COLONNES)
print(f"\n✅ {len(df_mediqal_final)} lignes converties depuis MediQAl OEQ")
print(f"   Dont avec priorité annotée : {df_mediqal_final['priorite'].notna().sum()}")

print("\nAperçu (2 exemples avec priorité) :")
print(df_mediqal_final[df_mediqal_final["priorite"].notna()][["instruction", "priorite", "niveau_confiance"]].head(2))

print("\nAperçu (2 exemples sans priorité) :")
print(df_mediqal_final[df_mediqal_final["priorite"].isna()][["instruction", "priorite", "niveau_confiance"]].head(2))


Nombre de cas avec priorité annotée disponible : 214

✅ 3954 lignes converties depuis MediQAl OEQ
   Dont avec priorité annotée : 214

Aperçu (2 exemples avec priorité) :
                                         instruction         priorite  \
0  Quels sont éléments de gravité au moment de l’...  urgence_moderee   
5  Que recherchez-vous à l’examen clinique pour t...  urgence_moderee   

      niveau_confiance  
0  annote_manuellement  
5  annote_manuellement  

Aperçu (2 exemples sans priorité) :
                                         instruction priorite  \
1  Caractérisez en une phrase le  tableau syndrom...      NaN   
2  Quelle est l'explication la plus probable de l...      NaN   

   niveau_confiance  
1  source_originale  
2  source_originale  


In [17]:
# CELLULE 14 - Conversion de FrenchMedMCQA vers le schéma unifié

import pandas as pd

# Recharger FrenchMedMCQA si besoin (normalement encore en mémoire, sinon décommenter)
# from datasets import load_dataset
# french_mcqa = load_dataset("nthngdy/frenchmedmcqa")
# df_frenchmcqa = pd.concat([french_mcqa["train"].to_pandas(), french_mcqa["validation"].to_pandas(), french_mcqa["test"].to_pandas()])

df_frenchmcqa = pd.concat([
    loaded_datasets["FrenchMedMCQA"]["train"].to_pandas(),
    loaded_datasets["FrenchMedMCQA"]["validation"].to_pandas(),
    loaded_datasets["FrenchMedMCQA"]["test"].to_pandas(),
], ignore_index=True)

print(f"Total FrenchMedMCQA (tous splits) : {len(df_frenchmcqa)}")

lettre_vers_col = {"a": "answer_a", "b": "answer_b", "c": "answer_c", "d": "answer_d", "e": "answer_e"}

def construire_reponse_mcqa(row):
    """Transforme les bonnes réponses en une réponse rédigée avec justification implicite."""
    correct = str(row["correct_answers"])
    lettres = [l.strip().lower() for l in correct.split(",") if l.strip()]
    
    reponses_correctes = []
    for l in lettres:
        col = lettre_vers_col.get(l)
        if col and pd.notna(row[col]):
            reponses_correctes.append(row[col])
    
    if not reponses_correctes:
        return None
    
    if len(reponses_correctes) == 1:
        return f"La réponse correcte est : {reponses_correctes[0]}"
    else:
        liste = "; ".join(reponses_correctes)
        return f"Les réponses correctes sont : {liste}"

def construire_instruction_mcqa(row):
    """Reconstruit la question avec les 5 options proposées."""
    options = []
    for lettre, col in lettre_vers_col.items():
        if pd.notna(row[col]):
            options.append(f"({lettre.upper()}) {row[col]}")
    return f"{row['question']}\n" + "\n".join(options)

lignes_frenchmcqa = []
skipped = 0

for _, row in df_frenchmcqa.iterrows():
    reponse = construire_reponse_mcqa(row)
    if reponse is None:
        skipped += 1
        continue
    
    lignes_frenchmcqa.append(nouvelle_ligne(
        langue="fr",
        source="FrenchMedMCQA",
        source_id=row["id"],
        instruction=construire_instruction_mcqa(row),
        contexte_patient=None,
        reponse=reponse,
        specialite=None,  # pas de spécialité fournie dans ce dataset
        priorite=None,
        niveau_confiance="source_originale"
    ))

df_frenchmcqa_final = pd.DataFrame(lignes_frenchmcqa, columns=SCHEMA_COLONNES)
print(f"\n✅ {len(df_frenchmcqa_final)} lignes converties (skipped: {skipped} sans réponse valide)")

print("\nAperçu :")
print(df_frenchmcqa_final[["instruction", "reponse"]].head(1).to_string())

Total FrenchMedMCQA (tous splits) : 1080

✅ 0 lignes converties (skipped: 1080 sans réponse valide)

Aperçu :
Empty DataFrame
Columns: [instruction, reponse]
Index: []


In [18]:
# CELLULE 14bis - Diagnostic du format de correct_answers

print("Types et valeurs uniques de correct_answers :")
print(df_frenchmcqa["correct_answers"].apply(type).value_counts())
print()
print(df_frenchmcqa["correct_answers"].unique()[:20])

print("\nExemple complet d'une ligne :")
print(df_frenchmcqa.iloc[0].to_dict())

Types et valeurs uniques de correct_answers :
correct_answers
<class 'int'>    1080
Name: count, dtype: int64

[4 2 1 3 0]

Exemple complet d'une ligne :
{'id': '230bac49b0fe863b772410bc8d01a025f63c3c999065480131d6334abd2efeff', 'question': 'Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha', 'answer_a': "Sont formées de noyaux d'hélium", 'answer_b': 'Sont peu pénétrantes', 'answer_c': "Toute l'énergie qu'elles transportent est cédée au long d'un parcours de quelques centimètres dans l'air", 'answer_d': 'Sont arrêtées par une feuille de papier', 'answer_e': 'Sont peu ionisantes', 'correct_answers': 4, 'number_correct_answers': 0}


In [19]:
# CELLULE 14 (corrigée) - Conversion de FrenchMedMCQA vers le schéma unifié

import pandas as pd

df_frenchmcqa = pd.concat([
    loaded_datasets["FrenchMedMCQA"]["train"].to_pandas(),
    loaded_datasets["FrenchMedMCQA"]["validation"].to_pandas(),
    loaded_datasets["FrenchMedMCQA"]["test"].to_pandas(),
], ignore_index=True)

print(f"Total FrenchMedMCQA (tous splits) : {len(df_frenchmcqa)}")

index_vers_col = {0: "answer_a", 1: "answer_b", 2: "answer_c", 3: "answer_d", 4: "answer_e"}
index_vers_lettre = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

def construire_reponse_mcqa(row):
    idx = row["correct_answers"]
    col = index_vers_col.get(idx)
    if col is None or pd.isna(row[col]):
        return None
    lettre = index_vers_lettre[idx]
    return f"La réponse correcte est ({lettre}) : {row[col]}"

def construire_instruction_mcqa(row):
    options = []
    for idx, col in index_vers_col.items():
        if pd.notna(row[col]):
            options.append(f"({index_vers_lettre[idx]}) {row[col]}")
    return f"{row['question']}\n" + "\n".join(options)

lignes_frenchmcqa = []
skipped = 0

for _, row in df_frenchmcqa.iterrows():
    reponse = construire_reponse_mcqa(row)
    if reponse is None:
        skipped += 1
        continue

    lignes_frenchmcqa.append(nouvelle_ligne(
        langue="fr",
        source="FrenchMedMCQA",
        source_id=row["id"],
        instruction=construire_instruction_mcqa(row),
        contexte_patient=None,
        reponse=reponse,
        specialite=None,
        priorite=None,
        niveau_confiance="source_originale"
    ))

df_frenchmcqa_final = pd.DataFrame(lignes_frenchmcqa, columns=SCHEMA_COLONNES)
print(f"\n✅ {len(df_frenchmcqa_final)} lignes converties (skipped: {skipped} sans réponse valide)")

print("\nAperçu :")
print(df_frenchmcqa_final[["instruction", "reponse"]].head(1).to_string())

Total FrenchMedMCQA (tous splits) : 1080

✅ 1080 lignes converties (skipped: 0 sans réponse valide)

Aperçu :
                                                                                                                                                                                                                                                                                                                                          instruction                                            reponse
0  Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha\n(A) Sont formées de noyaux d'hélium\n(B) Sont peu pénétrantes\n(C) Toute l'énergie qu'elles transportent est cédée au long d'un parcours de quelques centimètres dans l'air\n(D) Sont arrêtées par une feuille de papier\n(E) Sont peu ionisantes  La réponse correcte est (E) : Sont peu ionisantes


In [21]:
# CELLULE 15 - Conversion de MedQuAD vers le schéma unifié

import pandas as pd

df_medquad = loaded_datasets["MedQuAD"]["train"].to_pandas()
print(f"Total MedQuAD : {len(df_medquad)}")

# Nettoyage : retirer les lignes avec Question/Answer manquants ou vides
avant = len(df_medquad)
df_medquad_clean = df_medquad.dropna(subset=["Question", "Answer"])
df_medquad_clean = df_medquad_clean[
    (df_medquad_clean["Question"].astype(str).str.strip() != "") &
    (df_medquad_clean["Answer"].astype(str).str.strip() != "")
]
print(f"Lignes retirées (vides/manquantes) : {avant - len(df_medquad_clean)}")

lignes_medquad = []

for _, row in df_medquad_clean.iterrows():
    lignes_medquad.append(nouvelle_ligne(
        langue="en",
        source="MedQuAD",
        source_id=f"{row['qtype']}_{hash(row['Question']) % 100000}",  # pas d'id natif, on en génère un stable
        instruction=row["Question"],
        contexte_patient=None,
        reponse=row["Answer"],
        specialite=row["qtype"],  # qtype fait office de catégorie ici
        priorite=None,
        niveau_confiance="source_originale"
    ))

df_medquad_final = pd.DataFrame(lignes_medquad, columns=SCHEMA_COLONNES)
print(f"\n✅ {len(df_medquad_final)} lignes converties")

print("\nRépartition des qtype (top 10) :")
print(df_medquad_final["specialite"].value_counts().head(10))

print("\nAperçu :")
print(df_medquad_final[["instruction", "reponse"]].head(1).to_string())

Total MedQuAD : 16407
Lignes retirées (vides/manquantes) : 0

✅ 16407 lignes converties

Répartition des qtype (top 10) :
specialite
information        4535
symptoms           2748
treatment          2442
inheritance        1446
frequency          1120
genetic changes    1087
causes              727
exams and tests     653
research            395
outlook             361
Name: count, dtype: int64

Aperçu :
                                                instruction                                                                                                                                                                                                                                                                                                                                                                                                                                                                              reponse
0  Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?  L

In [22]:
# CELLULE 16 - Conversion d'un sous-échantillon UltraMedical-Preference pour le SFT

import pandas as pd

df_ultramed = loaded_datasets["UltraMedical-Preference (DPO)"]["train"].to_pandas()
print(f"Total UltraMedical-Preference (train) : {len(df_ultramed)}")

# Pour le SFT, on prend un échantillon aléatoire raisonnable (le reste sert au DPO)
N_ECHANTILLON_SFT = 800
echantillon_ultramed = df_ultramed.sample(n=N_ECHANTILLON_SFT, random_state=42).reset_index(drop=True)

def extraire_prompt_reponse(row):
    """Extrait le prompt utilisateur et la réponse 'chosen' (la meilleure réponse)."""
    prompt = row["prompt"]
    chosen = row["chosen"]
    # chosen est une liste de dicts [{role: user, content:...}, {role: assistant, content:...}]
    reponse_assistant = None
    for msg in chosen:
        if msg.get("role") == "assistant":
            reponse_assistant = msg.get("content")
            break
    return prompt, reponse_assistant

lignes_ultramed = []
skipped = 0

for _, row in echantillon_ultramed.iterrows():
    prompt, reponse = extraire_prompt_reponse(row)
    if not prompt or not reponse:
        skipped += 1
        continue

    lignes_ultramed.append(nouvelle_ligne(
        langue="en",
        source="UltraMedical-Preference",
        source_id=row["prompt_id"],
        instruction=prompt,
        contexte_patient=None,
        reponse=reponse,
        specialite=None,
        priorite=None,
        niveau_confiance="source_originale"
    ))

df_ultramed_final = pd.DataFrame(lignes_ultramed, columns=SCHEMA_COLONNES)
print(f"\n✅ {len(df_ultramed_final)} lignes converties pour le SFT (skipped: {skipped})")

print("\nAperçu (instruction tronquée) :")
apercu = df_ultramed_final.copy()
apercu["instruction"] = apercu["instruction"].str[:150]
apercu["reponse"] = apercu["reponse"].str[:150]
print(apercu[["instruction", "reponse"]].head(2).to_string())

Total UltraMedical-Preference (train) : 109353

✅ 800 lignes converties pour le SFT (skipped: 0)

Aperçu (instruction tronquée) :
                                                                                                                                              instruction                                                                                                                                                   reponse
0  In a 25-year-old male with a prior history of recurrent vomiting, now presenting with hematemesis and a confirmed endoscopic diagnosis of a Mallory-We  Let's assess the situation and consider the options available:\n\n1. **Patient Profile**: The patient is a 25-year-old male with a history of recurrent 
1  In managing a 32-year-old female with diabetes insipidus (DI) and dehydration, who is also presenting with nausea precluding oral intake, and consider    In managing a 32-year-old female with diabetes insipidus (DI) and dehydration, the primary goal i

In [23]:
# CELLULE 17 - Assemblage final du dataset SFT avec équilibrage

import pandas as pd

# Échantillonner MedQuAD pour ne pas écraser les autres sources
N_MEDQUAD = 700
df_medquad_sample = df_medquad_final.sample(n=N_MEDQUAD, random_state=42).reset_index(drop=True)

# Assembler toutes les sources
df_sft_final = pd.concat([
    df_mediqal_final,      # 3954 - FR - avec priorité
    df_frenchmcqa_final,   # 1080 - FR
    df_medquad_sample,     # 700  - EN (échantillonné)
    df_ultramed_final,     # 800  - EN
], ignore_index=True)

print(f"✅ Dataset SFT assemblé : {len(df_sft_final)} lignes")

print("\nRépartition par source :")
print(df_sft_final["source"].value_counts())

print("\nRépartition par langue :")
print(df_sft_final["langue"].value_counts())

print("\nRépartition par langue (%) :")
print((df_sft_final["langue"].value_counts(normalize=True) * 100).round(1))

print("\nNombre de lignes avec priorité annotée :")
print(df_sft_final["priorite"].notna().sum())

# Sauvegarde intermédiaire (avant split train/val/test)
import os
os.makedirs("data_processed", exist_ok=True)
df_sft_final.to_csv("data_processed/dataset_sft_assemble.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_assemble.csv")

✅ Dataset SFT assemblé : 6534 lignes

Répartition par source :
source
MediQAl_oeq                3954
FrenchMedMCQA              1080
UltraMedical-Preference     800
MedQuAD                     700
Name: count, dtype: int64

Répartition par langue :
langue
fr    5034
en    1500
Name: count, dtype: int64

Répartition par langue (%) :
langue
fr    77.0
en    23.0
Name: proportion, dtype: float64

Nombre de lignes avec priorité annotée :
214

💾 Sauvegardé -> data_processed/dataset_sft_assemble.csv


In [24]:
# CELLULE 18 - Split train / val / test

import pandas as pd
from sklearn.model_selection import train_test_split

df_sft_final = pd.read_csv("data_processed/dataset_sft_assemble.csv")

# Split 80% train / 10% val / 10% test, stratifié par source
train_df, temp_df = train_test_split(
    df_sft_final, test_size=0.2, random_state=42, stratify=df_sft_final["source"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["source"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

df_sft_split = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Train : {len(train_df)} ({len(train_df)/len(df_sft_split)*100:.1f}%)")
print(f"Val   : {len(val_df)} ({len(val_df)/len(df_sft_split)*100:.1f}%)")
print(f"Test  : {len(test_df)} ({len(test_df)/len(df_sft_split)*100:.1f}%)")

# Vérifier que les cas annotés priorité sont bien répartis dans les 3 splits (pas concentrés dans un seul)
print("\nRépartition des cas avec priorité annotée par split :")
print(df_sft_split[df_sft_split["priorite"].notna()]["split"].value_counts())

# Vérifier la répartition par source dans chaque split (contrôle de la stratification)
print("\nRépartition par source x split :")
print(pd.crosstab(df_sft_split["source"], df_sft_split["split"]))

df_sft_split.to_csv("data_processed/dataset_sft_split.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_split.csv")

Train : 5227 (80.0%)
Val   : 653 (10.0%)
Test  : 654 (10.0%)

Répartition des cas avec priorité annotée par split :
split
train    171
test      27
val       16
Name: count, dtype: int64

Répartition par source x split :
split                    test  train  val
source                                   
FrenchMedMCQA             108    864  108
MedQuAD                    70    560   70
MediQAl_oeq               396   3163  395
UltraMedical-Preference    80    640   80

💾 Sauvegardé -> data_processed/dataset_sft_split.csv


In [25]:
# CELLULE 19 - Test de l'anonymisation Presidio sur quelques exemples

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
import pandas as pd

# Initialiser les moteurs (FR + EN)
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

def anonymiser_texte(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(text=str(texte), language=lang_code)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur sur un texte : {e}")
        return texte

# Test sur 5 exemples avec des noms probables (Mme Barbie, M. Shoot vus plus tôt)
echantillon_test = df_sft_split[
    df_sft_split["contexte_patient"].str.contains("Mme|M\\.|Monsieur|Madame", na=False, regex=True)
].head(5)

print(f"Test sur {len(echantillon_test)} exemples contenant potentiellement des noms\n")

for idx, row in echantillon_test.iterrows():
    original = row["contexte_patient"][:300]
    anonymise = anonymiser_texte(row["contexte_patient"], row["langue"])[:300]
    print("="*80)
    print("AVANT :", original)
    print("-"*80)
    print("APRÈS :", anonymise)
    print()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.7 MB/s  0:03:430:00:0100:06
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Test sur 5 exemples contenant potentiellement des noms

⚠️ Erreur sur un texte : No matching recognizers were found to serve the request.
AVANT : Monsieur G., 85 ans est adressé aux urgences par sa maison de retraite pour un syndrome douloureux abdominal. Il a vomi à plusieurs reprises. Il a comme principaux antécédents une démence de type Alzheimer, une maladie de Parkinson évoluée, une hypertension artérielle, une gastrectomie pour perforat
--------------------------------------------------------------------------------
APRÈS : Monsieur G., 85 ans est adressé aux u

In [26]:
# CELLULE 19 (corrigée) - Configuration multilingue explicite de Presidio

from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
import pandas as pd

# Configuration explicite : quel modèle spaCy pour quelle langue
configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
        {"lang_code": "en", "model_name": "en_core_web_md"},
    ],
}

provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

# Analyzer configuré pour reconnaître fr ET en
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["fr", "en"])
anonymizer = AnonymizerEngine()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

def anonymiser_texte(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(text=str(texte), language=lang_code)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur sur un texte : {e}")
        return texte

# Re-test sur les mêmes 5 exemples
echantillon_test = df_sft_split[
    df_sft_split["contexte_patient"].str.contains("Mme|M\\.|Monsieur|Madame", na=False, regex=True)
].head(5)

print(f"Test sur {len(echantillon_test)} exemples\n")

for idx, row in echantillon_test.iterrows():
    original = row["contexte_patient"][:300]
    anonymise = anonymiser_texte(row["contexte_patient"], row["langue"])[:300]
    print("="*80)
    print("AVANT :", original)
    print("-"*80)
    print("APRÈS :", anonymise)
    print()

Test sur 5 exemples

AVANT : Monsieur G., 85 ans est adressé aux urgences par sa maison de retraite pour un syndrome douloureux abdominal. Il a vomi à plusieurs reprises. Il a comme principaux antécédents une démence de type Alzheimer, une maladie de Parkinson évoluée, une hypertension artérielle, une gastrectomie pour perforat
--------------------------------------------------------------------------------
APRÈS : <PERSON> G., 85 ans est adressé aux urgences par sa maison de retraite pour un syndrome douloureux abdominal. Il a vomi à plusieurs reprises. Il a comme principaux antécédents une démence de type <LOCATION>, une maladie de <PERSON> évoluée, une hypertension artérielle, une gastrectomie pour perforat

AVANT : Monsieur X., âgé de 65 ans, alcoolique est hospitalisé à la suite de l’apparition subite d’une fièvre élevée (40°C) associée à douleur thoracique, dyspnée et expectoration purulente rouillée.\n L’examen clinique et en particulier l’auscultation révèle des ra

In [27]:
# CELLULE 19 (v2) - Presidio corrigé : exclusion des termes médicaux + détection renforcée

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
import pandas as pd
import re

# ============================================================
# 1. Liste de termes médicaux à NE JAMAIS anonymiser
#    (maladies éponymes, syndromes portant un nom propre)
# ============================================================
TERMES_MEDICAUX_PROTEGES = {
    "alzheimer", "parkinson", "fallot", "eisenmenger", "yacoub", "roger",
    "glasgow", "manchester", "french", "rashkind", "norwood", "loxen",
    "glucophage", "prostaglandine", "gcs", "nyha", "covid", "sars",
    "crohn", "basedow", "hodgkin", "kawasaki", "willebrand", "raynaud",
    "cushing", "addison", "wolff", "parkinson-white", "ebstein",
}

configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
        {"lang_code": "en", "model_name": "en_core_web_md"},
    ],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["fr", "en"])

# ============================================================
# 2. Recognizer additionnel pour capturer explicitement
#    "Monsieur X", "Madame Y.M.", "M. D.", "Mme B." etc.
# ============================================================
pattern_civilite = Pattern(
    name="civilite_initiale",
    regex=r"\b(Monsieur|Madame|Mademoiselle|M\.|Mme|Mlle)\s+[A-Z][a-zA-Z]*(\.[A-Z]\.?)?",
    score=0.85
)
recognizer_civilite_fr = PatternRecognizer(
    supported_entity="PERSON",
    patterns=[pattern_civilite],
    supported_language="fr"
)
analyzer.registry.add_recognizer(recognizer_civilite_fr)

anonymizer = AnonymizerEngine()

def anonymiser_texte(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(text=str(texte), language=lang_code)

        # Filtrer : retirer les détections qui matchent un terme médical protégé
        resultats_filtres = []
        for r in resultats:
            texte_detecte = str(texte)[r.start:r.end].lower().strip(" .,")
            if texte_detecte in TERMES_MEDICAUX_PROTEGES:
                continue  # on ignore cette détection, c'est un terme médical légitime
            resultats_filtres.append(r)

        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

# ============================================================
# Re-test sur les mêmes 5 exemples + un exemple avec Alzheimer/Parkinson
# ============================================================
df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

echantillon_test = df_sft_split[
    df_sft_split["contexte_patient"].str.contains("Mme|M\\.|Monsieur|Madame", na=False, regex=True)
].head(5)

print(f"Test sur {len(echantillon_test)} exemples\n")

for idx, row in echantillon_test.iterrows():
    original = row["contexte_patient"][:300]
    anonymise = anonymiser_texte(row["contexte_patient"], row["langue"])[:300]
    print("="*80)
    print("AVANT :", original)
    print("-"*80)
    print("APRÈS :", anonymise)
    print()

Test sur 5 exemples

AVANT : Monsieur G., 85 ans est adressé aux urgences par sa maison de retraite pour un syndrome douloureux abdominal. Il a vomi à plusieurs reprises. Il a comme principaux antécédents une démence de type Alzheimer, une maladie de Parkinson évoluée, une hypertension artérielle, une gastrectomie pour perforat
--------------------------------------------------------------------------------
APRÈS : <PERSON>., 85 ans est adressé aux urgences par sa maison de retraite pour un syndrome douloureux abdominal. Il a vomi à plusieurs reprises. Il a comme principaux antécédents une démence de type Alzheimer, une maladie de Parkinson évoluée, une hypertension artérielle, une gastrectomie pour perforatio

AVANT : Monsieur X., âgé de 65 ans, alcoolique est hospitalisé à la suite de l’apparition subite d’une fièvre élevée (40°C) associée à douleur thoracique, dyspnée et expectoration purulente rouillée.\n L’examen clinique et en particulier l’auscultation révèle des ra

In [28]:
# CELLULE 20 - Application de l'anonymisation sur tout le dataset

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

print(f"Anonymisation de {len(df_sft_split)} lignes sur 3 colonnes (instruction, contexte_patient, reponse)...")
print("Ça peut prendre plusieurs minutes, sois patient.\n")

df_sft_anonymise = df_sft_split.copy()

print("Colonne 1/3 : instruction")
df_sft_anonymise["instruction"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte(row["instruction"], row["langue"]), axis=1
)

print("\nColonne 2/3 : contexte_patient")
df_sft_anonymise["contexte_patient"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte(row["contexte_patient"], row["langue"]), axis=1
)

print("\nColonne 3/3 : reponse")
df_sft_anonymise["reponse"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte(row["reponse"], row["langue"]), axis=1
)

df_sft_anonymise.to_csv("data_processed/dataset_sft_anonymise.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_anonymise.csv")

# Vérification rapide : combien de lignes contiennent encore un tag <PERSON> etc. (normal, c'est voulu)
import re
pattern_tags = re.compile(r"<[A-Z_]+>")
nb_lignes_avec_masquage = df_sft_anonymise["contexte_patient"].astype(str).str.contains(pattern_tags, regex=True).sum()
print(f"\nLignes où au moins une entité a été masquée dans contexte_patient : {nb_lignes_avec_masquage} / {len(df_sft_anonymise)}")

Anonymisation de 6534 lignes sur 3 colonnes (instruction, contexte_patient, reponse)...
Ça peut prendre plusieurs minutes, sois patient.

Colonne 1/3 : instruction


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 2/3 : contexte_patient


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 3/3 : reponse


  0%|          | 0/6534 [00:00<?, ?it/s]


💾 Sauvegardé -> data_processed/dataset_sft_anonymise.csv

Lignes où au moins une entité a été masquée dans contexte_patient : 3277 / 6534


In [29]:
# CELLULE 21 - Contrôle qualité de l'anonymisation (échantillon aléatoire)

import pandas as pd

df_final = pd.read_csv("data_processed/dataset_sft_anonymise.csv")

# Échantillon aléatoire de 10 lignes anonymisées pour relecture manuelle
echantillon_qc = df_final[df_final["contexte_patient"].notna()].sample(n=10, random_state=7)

print("Échantillon de contrôle qualité (relecture manuelle recommandée) :\n")
for idx, row in echantillon_qc.iterrows():
    print("="*80)
    print(f"Source: {row['source']} | Langue: {row['langue']}")
    print(str(row['contexte_patient'])[:400])
    print()

# Statistiques sur les types d'entités masquées (pour documenter dans le rapport)
import re
from collections import Counter

pattern_tags = re.compile(r"<([A-Z_]+)>")
compteur_entites = Counter()

for col in ["instruction", "contexte_patient", "reponse"]:
    for texte in df_final[col].dropna().astype(str):
        tags = pattern_tags.findall(texte)
        compteur_entites.update(tags)

print("\n" + "="*80)
print("Types d'entités masquées (toutes colonnes confondues) :")
for entite, count in compteur_entites.most_common():
    print(f"  {entite}: {count}")

Échantillon de contrôle qualité (relecture manuelle recommandée) :

Source: MediQAl_oeq | Langue: fr
<PERSON>, agé de 86 ans , est suivi pour une bronchite chronique depuis 10 ans. Il a arrêté de fumer depuis de nombreuses années et consomme un verre de vin rouge par jour. Dans les antécédents, on note il y a 22 ans un grave accident de la route ayant nécessité une transfusion sanguine. Une cirrhose avec ictère et ascite modérée a été récemment découverte associée à une hépatomégalie.
 Les résult

Source: MediQAl_oeq | Langue: fr
<PERSON>, 74 ans vous est adressé en consultation par son médecin traitant pour dyspnée progressivement croissante sur les dernières semaines. Ce patient, retraité depuis une dizaine d’années a travaillé dans le bâtiment, puis en fin de carrière comme gardien d’immeuble. Pendant ces années-là, il fumait la pipe, avec une consommation de 70 grammes par semaine pendant 10 ans. Il n’a aucun antécéden

Source: MediQAl_oeq | Langue: fr
Un homme âgé de 42 ans présen

In [30]:
# CELLULE 22 - Diagnostic : que détecte-t-on comme LOCATION/ORGANIZATION ?

import pandas as pd
from collections import Counter

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

# On ré-analyse (SANS anonymiser) un échantillon pour voir le texte exact détecté
echantillon = df_sft_split[df_sft_split["contexte_patient"].notna()].sample(n=100, random_state=1)

detections_location = Counter()
detections_organization = Counter()
detections_person = Counter()

for _, row in echantillon.iterrows():
    texte = str(row["contexte_patient"])
    lang = "fr" if row["langue"] == "fr" else "en"
    resultats = analyzer.analyze(text=texte, language=lang)
    for r in resultats:
        span = texte[r.start:r.end]
        if r.entity_type == "LOCATION":
            detections_location[span] += 1
        elif r.entity_type == "ORGANIZATION":
            detections_organization[span] += 1
        elif r.entity_type == "PERSON":
            detections_person[span] += 1

print("Termes les plus fréquemment détectés comme LOCATION (faux positifs à identifier) :")
for terme, count in detections_location.most_common(30):
    print(f"  '{terme}': {count}")

print("\nTermes les plus fréquemment détectés comme ORGANIZATION :")
for terme, count in detections_organization.most_common(30):
    print(f"  '{terme}': {count}")

print("\nTermes les plus fréquemment détectés comme PERSON :")
for terme, count in detections_person.most_common(30):
    print(f"  '{terme}': {count}")

Termes les plus fréquemment détectés comme LOCATION (faux positifs à identifier) :
  'G': 12
  'Hg': 8
  'Monocytes': 8
  'Lymphocytes': 7
  'Gram': 5
  'ALAT': 4
  'France': 4
  'TA': 4
  'cyanosé': 4
  'la France': 4
  'Globules': 4
  'Hémoglobine': 4
  'N': 3
  'Pl Bicarbonate': 3
  'Ln': 3
  'Créatinine': 3
  'Sud': 3
  'Na': 2
  'K': 2
  'L
 ': 2
  'hospitalisée': 2
  'André': 2
  'Hémogramme': 2
  'Sg': 2
  'Glasgow': 2
  'Sg Erythrocytes': 2
  'Urgences': 2
  'Roubaix': 2
  'O2': 2
  'traitée': 2

Termes les plus fréquemment détectés comme ORGANIZATION :
  'ASAT': 7
  'Formule': 4
  'ACUITEL@': 4
  'ISAGA': 3
  'Pl CO2 total': 2
  'LDH': 2
  'SA': 2
  'SAMU': 2
  'SGOT': 2
  'abandonné': 2
  'ELISA': 2
  'IgM': 2
  'CPK': 1
  'CRP': 1
  'E.P.': 1
  'Hébra': 1
  'Pl Créatinine': 1
  'ORL': 1
  'G/L dont polynucléaires neutrophiles 0,85': 1
  'G/L': 1
  'IDEOS': 1
  'MOPRAL®': 1
  'TEMESTA': 1
  'CLE': 1
  'Léri': 1
  'RCP': 1
  'Pl TCA': 1
  'G/l': 1
  'SaO2': 1
  'alpha1':

In [31]:
# CELLULE 23 - Anonymisation recentrée : PERSON uniquement (LOCATION/ORGANIZATION trop bruités sur ce corpus)

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
import pandas as pd

configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
        {"lang_code": "en", "model_name": "en_core_web_md"},
    ],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["fr", "en"])

# Recognizer par motif pour les civilités (fiable et précis)
pattern_civilite = Pattern(
    name="civilite_initiale",
    regex=r"\b(Monsieur|Madame|Mademoiselle|M\.|Mme|Mlle|Mr)\s+[A-Z][a-zA-Z]*(\.[A-Z]\.?)?",
    score=0.9
)
recognizer_civilite_fr = PatternRecognizer(
    supported_entity="PERSON", patterns=[pattern_civilite], supported_language="fr"
)
analyzer.registry.add_recognizer(recognizer_civilite_fr)

anonymizer = AnonymizerEngine()

# Entités à conserver : uniquement les données réellement identifiantes/sensibles
ENTITES_CIBLEES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IP_ADDRESS", "US_DRIVER_LICENSE", "UK_NHS"]
SEUIL_CONFIANCE = 0.6  # on relève le seuil pour réduire les faux positifs

def anonymiser_texte_v2(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(
            text=str(texte), language=lang_code, entities=ENTITES_CIBLEES
        )
        resultats_filtres = [r for r in resultats if r.score >= SEUIL_CONFIANCE]
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

# Re-test sur les mêmes 10 exemples problématiques vus en Cellule 21
df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")
echantillon_qc = df_sft_split[df_sft_split["contexte_patient"].notna()].sample(n=10, random_state=7)

print("Nouveau test avec entités ciblées (PERSON + contacts uniquement) :\n")
for idx, row in echantillon_qc.iterrows():
    original = str(row["contexte_patient"])[:400]
    anonymise = anonymiser_texte_v2(row["contexte_patient"], row["langue"])[:400]
    print("="*80)
    print("AVANT :", original)
    print("-"*80)
    print("APRÈS :", anonymise)
    print()

Nouveau test avec entités ciblées (PERSON + contacts uniquement) :

AVANT : Mr X, agé de 86 ans , est suivi pour une bronchite chronique depuis 10 ans. Il a arrêté de fumer depuis de nombreuses années et consomme un verre de vin rouge par jour. Dans les antécédents, on note il y a 22 ans un grave accident de la route ayant nécessité une transfusion sanguine. Une cirrhose avec ictère et ascite modérée a été récemment découverte associée à une hépatomégalie.
 Les résultats 
--------------------------------------------------------------------------------
APRÈS : <PERSON>, agé de 86 ans , est suivi pour une bronchite chronique depuis 10 ans. Il a arrêté de fumer depuis de nombreuses années et consomme un verre de vin rouge par jour. Dans les antécédents, on note il y a 22 ans un grave accident de la route ayant nécessité une transfusion sanguine. Une cirrhose avec ictère et ascite modérée a été récemment découverte associée à une hépatomégalie.
 Les résult

AVANT : Monsieur F., 74 ans vous

In [32]:
# CELLULE 23bis - Ajout des noms de techniques chirurgicales/médicales éponymes à exclure

TERMES_MEDICAUX_PROTEGES = {
    # Maladies/syndromes déjà identifiés
    "alzheimer", "parkinson", "fallot", "eisenmenger", "yacoub", "roger",
    "glasgow", "manchester", "french", "kawasaki", "willebrand", "raynaud",
    "cushing", "addison", "wolff", "ebstein", "crohn", "basedow", "hodgkin",
    # Techniques chirurgicales cardiaques (très fréquentes dans ce corpus)
    "rashkind", "norwood", "blalock", "blalock-taussig", "blalock-hanlon",
    "fontan", "glenn", "mustard", "senning", "damus-kaye-stansel", "ross",
    "konno", "nikaidoh", "rastelli", "jatene",
}

def anonymiser_texte_v3(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(
            text=str(texte), language=lang_code, entities=ENTITES_CIBLEES
        )
        resultats_filtres = []
        for r in resultats:
            if r.score < SEUIL_CONFIANCE:
                continue
            span = str(texte)[r.start:r.end].lower().strip(" .,")
            # Vérifier si le terme détecté (ou une partie via hyphen) est un terme protégé
            if span in TERMES_MEDICAUX_PROTEGES or any(part in TERMES_MEDICAUX_PROTEGES for part in span.split("-")):
                continue
            resultats_filtres.append(r)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

# Re-test rapide sur le cas Blalock-Hanlon
test_texte = "Mme G. âgée de 26 ans a eu une correction atriale après une intervention de Blalock-Hanlon néonatale."
print("AVANT :", test_texte)
print("APRÈS :", anonymiser_texte_v3(test_texte, "fr"))

AVANT : Mme G. âgée de 26 ans a eu une correction atriale après une intervention de Blalock-Hanlon néonatale.
APRÈS : <PERSON> âgée de 26 ans a eu une correction atriale après une intervention de Blalock-Hanlon néonatale.


In [33]:
# CELLULE 24 - Application finale de l'anonymisation (v3) sur tout le dataset

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

print(f"Anonymisation finale de {len(df_sft_split)} lignes (PERSON + contacts, avec exclusions médicales)...\n")

df_sft_anonymise = df_sft_split.copy()

print("Colonne 1/3 : instruction")
df_sft_anonymise["instruction"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte_v3(row["instruction"], row["langue"]), axis=1
)

print("\nColonne 2/3 : contexte_patient")
df_sft_anonymise["contexte_patient"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte_v3(row["contexte_patient"], row["langue"]), axis=1
)

print("\nColonne 3/3 : reponse")
df_sft_anonymise["reponse"] = df_sft_anonymise.progress_apply(
    lambda row: anonymiser_texte_v3(row["reponse"], row["langue"]), axis=1
)

df_sft_anonymise.to_csv("data_processed/dataset_sft_anonymise_final.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final.csv")

# Statistiques finales
import re
pattern_tags = re.compile(r"<PERSON>|<EMAIL_ADDRESS>|<PHONE_NUMBER>")
nb_lignes_masquees = df_sft_anonymise["contexte_patient"].astype(str).str.contains(pattern_tags, regex=True).sum()
print(f"\nLignes avec masquage PERSON/contact dans contexte_patient : {nb_lignes_masquees} / {len(df_sft_anonymise)}")

# Aperçu final de contrôle (5 exemples au hasard)
print("\n--- Aperçu final de contrôle ---")
apercu = df_sft_anonymise[df_sft_anonymise["contexte_patient"].notna()].sample(5, random_state=99)
for idx, row in apercu.iterrows():
    print("="*80)
    print(str(row["contexte_patient"])[:300])

Anonymisation finale de 6534 lignes (PERSON + contacts, avec exclusions médicales)...

Colonne 1/3 : instruction


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 2/3 : contexte_patient


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 3/3 : reponse


  0%|          | 0/6534 [00:00<?, ?it/s]


💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final.csv

Lignes avec masquage PERSON/contact dans contexte_patient : 2394 / 6534

--- Aperçu final de contrôle ---
Une patiente de 68 ans est amenée aux Urgences par le SAMU pour troubles de conscience et détresse respiratoire.L’histoire de la maladie a débuté le 23 mai par un tableau pseudo-grippal associant asthénie et myalgies. La patiente n’a pas mesuré sa température. Elle a pris du paracétamol pour les dou
Un enfant de 15 mois est adressé en consultation pour hématomes fréquents depuis qu’il commence à marcher. Son grand père maternel est décédé de complications hémorragiques après sa chute d’une échelle, la mère en revanche a déjà été opérée des amygdales sans problème dans son enfance.
 Après avoir 
<PERSON> 58 ans, 180 cm, 72 kg, artiste peintre se plaint de tremblements et d’un ralentissement de ses mouvements volontaires qui le gênent dans l’exercice de son activité ainsi que de vertiges lorsqu’il se lève le matin. I

In [1]:
from huggingface_hub import login
import getpass

token = getpass.getpass("Colle ton token Hugging Face (write) : ")
login(token=token)

print("✅ Authentification réussie")

Colle ton token Hugging Face (write) :  ········


✅ Authentification réussie


In [3]:
# CELLULE 26 - Upload du dataset vers Hugging Face Hub

import pandas as pd
from datasets import Dataset, DatasetDict

df_final = pd.read_csv("data_processed/dataset_sft_anonymise_final.csv")

# Reconstruire les 3 splits séparément à partir de la colonne 'split'
train_ds = Dataset.from_pandas(df_final[df_final["split"] == "train"].drop(columns=["split"]).reset_index(drop=True))
val_ds = Dataset.from_pandas(df_final[df_final["split"] == "val"].drop(columns=["split"]).reset_index(drop=True))
test_ds = Dataset.from_pandas(df_final[df_final["split"] == "test"].drop(columns=["split"]).reset_index(drop=True))

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds,
})

print(dataset_dict)

# ⚠️ Remplace par ton propre nom d'utilisateur HuggingFace
NOM_UTILISATEUR = "UserMarrakech"
NOM_DATASET = "chsa-triage-medical-sft"

repo_id = f"{NOM_UTILISATEUR}/{NOM_DATASET}"

dataset_dict.push_to_hub(repo_id, private=True)  # private=True recommandé pour données médicales, même anonymisées

print(f"\n✅ Dataset uploadé sur https://huggingface.co/datasets/{repo_id}")

DatasetDict({
    train: Dataset({
        features: ['id', 'langue', 'source', 'source_id', 'instruction', 'contexte_patient', 'reponse', 'specialite', 'priorite', 'niveau_confiance'],
        num_rows: 5227
    })
    validation: Dataset({
        features: ['id', 'langue', 'source', 'source_id', 'instruction', 'contexte_patient', 'reponse', 'specialite', 'priorite', 'niveau_confiance'],
        num_rows: 653
    })
    test: Dataset({
        features: ['id', 'langue', 'source', 'source_id', 'instruction', 'contexte_patient', 'reponse', 'specialite', 'priorite', 'niveau_confiance'],
        num_rows: 654
    })
})


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Dataset uploadé sur https://huggingface.co/datasets/UserMarrakech/chsa-triage-medical-sft


In [4]:
# CELLULE 23ter (LOCAL) - Liste d'exclusion élargie + ré-anonymisation complète

TERMES_MEDICAUX_PROTEGES = {
    # Déjà présents
    "alzheimer", "parkinson", "fallot", "eisenmenger", "yacoub", "roger",
    "glasgow", "manchester", "french", "kawasaki", "willebrand", "raynaud",
    "cushing", "addison", "wolff", "ebstein", "crohn", "basedow", "hodgkin",
    "rashkind", "norwood", "blalock", "blalock-taussig", "blalock-hanlon",
    "fontan", "glenn", "mustard", "senning", "damus-kaye-stansel", "ross",
    "konno", "nikaidoh", "rastelli", "jatene",
    # Ajout : mucopolysaccharidoses et maladies génétiques éponymes (EN/FR)
    "hurler", "hunter", "scheie", "morquio", "sanfilippo", "maroteaux-lamy",
    "sly", "gaucher", "fabry", "pompe", "niemann-pick", "tay-sachs",
    "marfan", "ehlers-danlos", "turner", "klinefelter", "down",
    "prader-willi", "angelman", "rett", "duchenne", "becker",
    "charcot-marie-tooth", "guillain-barré", "creutzfeldt-jakob",
    "huntington", "friedreich", "wilson", "menkes",
}

print(f"Liste d'exclusion mise à jour : {len(TERMES_MEDICAUX_PROTEGES)} termes")

def anonymiser_texte_v4(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(
            text=str(texte), language=lang_code, entities=ENTITES_CIBLEES
        )
        resultats_filtres = []
        for r in resultats:
            if r.score < SEUIL_CONFIANCE:
                continue
            span = str(texte)[r.start:r.end].lower().strip(" .,")
            if span in TERMES_MEDICAUX_PROTEGES or any(part in TERMES_MEDICAUX_PROTEGES for part in span.split("-")):
                continue
            resultats_filtres.append(r)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

# Test de vérification rapide sur le cas Hurler
test_texte = "Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder."
print("\nAVANT :", test_texte)
print("APRÈS :", anonymiser_texte_v4(test_texte, "en"))

Liste d'exclusion mise à jour : 63 termes

AVANT : Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder.
⚠️ Erreur : name 'analyzer' is not defined
APRÈS : Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder.


In [2]:
# CELLULE 23ter (LOCAL, corrigée) - Rechargement complet de Presidio + liste élargie

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
import pandas as pd

# Recharger le moteur NLP multilingue
configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
        {"lang_code": "en", "model_name": "en_core_web_md"},
    ],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["fr", "en"])

# Recognizer civilités
pattern_civilite = Pattern(
    name="civilite_initiale",
    regex=r"\b(Monsieur|Madame|Mademoiselle|M\.|Mme|Mlle|Mr)\s+[A-Z][a-zA-Z]*(\.[A-Z]\.?)?",
    score=0.9
)
recognizer_civilite_fr = PatternRecognizer(
    supported_entity="PERSON", patterns=[pattern_civilite], supported_language="fr"
)
analyzer.registry.add_recognizer(recognizer_civilite_fr)

anonymizer = AnonymizerEngine()

ENTITES_CIBLEES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IP_ADDRESS", "US_DRIVER_LICENSE", "UK_NHS"]
SEUIL_CONFIANCE = 0.6

# Liste d'exclusion élargie
TERMES_MEDICAUX_PROTEGES = {
    "alzheimer", "parkinson", "fallot", "eisenmenger", "yacoub", "roger",
    "glasgow", "manchester", "french", "kawasaki", "willebrand", "raynaud",
    "cushing", "addison", "wolff", "ebstein", "crohn", "basedow", "hodgkin",
    "rashkind", "norwood", "blalock", "blalock-taussig", "blalock-hanlon",
    "fontan", "glenn", "mustard", "senning", "damus-kaye-stansel", "ross",
    "konno", "nikaidoh", "rastelli", "jatene",
    "hurler", "hunter", "scheie", "morquio", "sanfilippo", "maroteaux-lamy",
    "sly", "gaucher", "fabry", "pompe", "niemann-pick", "tay-sachs",
    "marfan", "ehlers-danlos", "turner", "klinefelter", "down",
    "prader-willi", "angelman", "rett", "duchenne", "becker",
    "charcot-marie-tooth", "guillain-barré", "creutzfeldt-jakob",
    "huntington", "friedreich", "wilson", "menkes",
}

def anonymiser_texte_v4(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(text=str(texte), language=lang_code, entities=ENTITES_CIBLEES)
        resultats_filtres = []
        for r in resultats:
            if r.score < SEUIL_CONFIANCE:
                continue
            span = str(texte)[r.start:r.end].lower().strip(" .,")
            if span in TERMES_MEDICAUX_PROTEGES or any(part in TERMES_MEDICAUX_PROTEGES for part in span.split("-")):
                continue
            resultats_filtres.append(r)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

test_texte = "Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder."
print("AVANT :", test_texte)
print("APRÈS :", anonymiser_texte_v4(test_texte, "en"))

AVANT : Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder.
APRÈS : Mucopolysaccharidosis type I (Hurler syndrome), also known as Hurler-Scheie syndrome, is a rare genetic disorder.


In [3]:
# CELLULE 24bis (LOCAL) - Ré-anonymisation complète avec liste élargie

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

# On repart du fichier SPLITTÉ (pas encore anonymisé) pour éviter le double traitement
df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

print(f"Ré-anonymisation de {len(df_sft_split)} lignes avec la liste élargie...\n")

df_sft_anonymise_v2 = df_sft_split.copy()

print("Colonne 1/3 : instruction")
df_sft_anonymise_v2["instruction"] = df_sft_anonymise_v2.progress_apply(
    lambda row: anonymiser_texte_v4(row["instruction"], row["langue"]), axis=1
)

print("\nColonne 2/3 : contexte_patient")
df_sft_anonymise_v2["contexte_patient"] = df_sft_anonymise_v2.progress_apply(
    lambda row: anonymiser_texte_v4(row["contexte_patient"], row["langue"]), axis=1
)

print("\nColonne 3/3 : reponse")
df_sft_anonymise_v2["reponse"] = df_sft_anonymise_v2.progress_apply(
    lambda row: anonymiser_texte_v4(row["reponse"], row["langue"]), axis=1
)

df_sft_anonymise_v2.to_csv("data_processed/dataset_sft_anonymise_final_v2.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v2.csv")

# Vérification : le cas Hurler (UltraMedical-Preference) est-il maintenant correct ?
exemple_hurler = df_sft_anonymise_v2[df_sft_anonymise_v2["reponse"].str.contains("Hurler", na=False)]
print(f"\nLignes contenant 'Hurler' (doivent être préservées) : {len(exemple_hurler)}")
if len(exemple_hurler) > 0:
    print(exemple_hurler.iloc[0]["reponse"][:200])

Ré-anonymisation de 6534 lignes avec la liste élargie...

Colonne 1/3 : instruction


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 2/3 : contexte_patient


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 3/3 : reponse


  0%|          | 0/6534 [00:00<?, ?it/s]


💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v2.csv

Lignes contenant 'Hurler' (doivent être préservées) : 2
Mucopolysaccharidosis type I (<PERSON> I), also known as Hurler syndrome, of Hurler-Scheie syndrome, is a rare genetic disorder caused by a deficiency of the enzyme alpha-L-iduronidase. This deficienc


In [4]:
# CELLULE 24ter (LOCAL) - Correction du filtre : vérifier mot par mot, pas juste le span entier

def contient_terme_protege(span_lower):
    """Vérifie si un des mots du span (séparé par espace ou tiret) est un terme médical protégé."""
    mots = span_lower.replace("-", " ").split()
    return any(mot in TERMES_MEDICAUX_PROTEGES for mot in mots)

def anonymiser_texte_v5(texte, langue):
    if pd.isna(texte) or texte == "":
        return texte
    lang_code = "fr" if langue == "fr" else "en"
    try:
        resultats = analyzer.analyze(text=str(texte), language=lang_code, entities=ENTITES_CIBLEES)
        resultats_filtres = []
        for r in resultats:
            if r.score < SEUIL_CONFIANCE:
                continue
            span = str(texte)[r.start:r.end].lower().strip(" .,")
            if contient_terme_protege(span):
                continue
            resultats_filtres.append(r)
        anonymise = anonymizer.anonymize(text=str(texte), analyzer_results=resultats_filtres)
        return anonymise.text
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return texte

# Test ciblé sur "Hurler I"
test_texte = "Mucopolysaccharidosis type I (Hurler I), also known as Hurler syndrome, of Hurler-Scheie syndrome."
print("AVANT :", test_texte)
print("APRÈS :", anonymiser_texte_v5(test_texte, "en"))

AVANT : Mucopolysaccharidosis type I (Hurler I), also known as Hurler syndrome, of Hurler-Scheie syndrome.
APRÈS : Mucopolysaccharidosis type I (Hurler I), also known as Hurler syndrome, of Hurler-Scheie syndrome.


In [5]:
# CELLULE 24quater (LOCAL) - Anonymisation finale (v5) sur tout le dataset

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

print(f"Anonymisation finale (v5) de {len(df_sft_split)} lignes...\n")

df_sft_final_v3 = df_sft_split.copy()

print("Colonne 1/3 : instruction")
df_sft_final_v3["instruction"] = df_sft_final_v3.progress_apply(
    lambda row: anonymiser_texte_v5(row["instruction"], row["langue"]), axis=1
)

print("\nColonne 2/3 : contexte_patient")
df_sft_final_v3["contexte_patient"] = df_sft_final_v3.progress_apply(
    lambda row: anonymiser_texte_v5(row["contexte_patient"], row["langue"]), axis=1
)

print("\nColonne 3/3 : reponse")
df_sft_final_v3["reponse"] = df_sft_final_v3.progress_apply(
    lambda row: anonymiser_texte_v5(row["reponse"], row["langue"]), axis=1
)

df_sft_final_v3.to_csv("data_processed/dataset_sft_anonymise_final_v3.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v3.csv")

# Vérification finale
exemple_hurler = df_sft_final_v3[df_sft_final_v3["reponse"].str.contains("Hurler I", na=False)]
print(f"\nLignes contenant 'Hurler I' intact : {len(exemple_hurler)}")

Anonymisation finale (v5) de 6534 lignes...

Colonne 1/3 : instruction


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 2/3 : contexte_patient


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 3/3 : reponse


  0%|          | 0/6534 [00:00<?, ?it/s]


💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v3.csv

Lignes contenant 'Hurler I' intact : 0


In [6]:
# Vérification élargie

df_check = pd.read_csv("data_processed/dataset_sft_anonymise_final_v3.csv")

# Chercher toutes les lignes contenant "Hurler" (sans restriction) et toutes celles avec <PERSON>
lignes_hurler = df_check[df_check["reponse"].str.contains("Hurler", na=False)]
print(f"Lignes contenant 'Hurler' (n'importe où) : {len(lignes_hurler)}")

lignes_person_pres_hurler = df_check[df_check["reponse"].str.contains("PERSON", na=False) & df_check["reponse"].str.contains("Hurler", na=False)]
print(f"Lignes avec à la fois 'Hurler' ET '<PERSON>' (signe de faux positif résiduel) : {len(lignes_person_pres_hurler)}")

if len(lignes_hurler) > 0:
    print("\nExtrait de la première ligne trouvée :")
    print(lignes_hurler.iloc[0]["reponse"][:300])

Lignes contenant 'Hurler' (n'importe où) : 2
Lignes avec à la fois 'Hurler' ET '<PERSON>' (signe de faux positif résiduel) : 1

Extrait de la première ligne trouvée :
Mucopolysaccharidosis type I (<PERSON> I), also known as Hurler syndrome, of Hurler-Scheie syndrome, is a rare genetic disorder caused by a deficiency of the enzyme alpha-L-iduronidase. This deficiency leads to the accumulation of glycosaminoglycans (GAGs) in cells, tissues, and organs, causing a ra


In [7]:
# Diagnostic : quel span exact Presidio détecte-t-il sur ce texte précis ?

texte_probleme = "Mucopolysaccharidosis type I (Hurler I), also known as Hurler syndrome, of Hurler-Scheie syndrome, is a rare genetic disorder"

resultats = analyzer.analyze(text=texte_probleme, language="en", entities=ENTITES_CIBLEES)
for r in resultats:
    span_brut = texte_probleme[r.start:r.end]
    span_nettoye = span_brut.lower().strip(" .,")
    print(f"Span détecté : '{span_brut}' | Score: {r.score:.2f}")
    print(f"  -> Après nettoyage : '{span_nettoye}'")
    print(f"  -> Mots après split : {span_nettoye.replace('-', ' ').split()}")
    print(f"  -> Protégé ? {contient_terme_protege(span_nettoye)}")
    print()

Span détecté : 'Hurler' | Score: 0.85
  -> Après nettoyage : 'hurler'
  -> Mots après split : ['hurler']
  -> Protégé ? True



In [8]:
# Diagnostic sur le VRAI texte complet (pas un extrait manuel)

df_original = pd.read_csv("data_processed/dataset_sft_split.csv")
ligne_hurler = df_original[df_original["reponse"].str.contains("Hurler", na=False)].iloc[0]
texte_complet = ligne_hurler["reponse"]

print(f"Longueur du texte : {len(texte_complet)} caractères\n")

resultats = analyzer.analyze(text=texte_complet, language="en", entities=ENTITES_CIBLEES)

print(f"Nombre total de détections : {len(resultats)}\n")

for r in resultats:
    span_brut = texte_complet[r.start:r.end]
    span_nettoye = span_brut.lower().strip(" .,")
    protege = contient_terme_protege(span_nettoye)
    marqueur = "✅ PROTÉGÉ" if protege else "❌ SERA MASQUÉ"
    print(f"{marqueur} | Span: '{span_brut}' | Score: {r.score:.2f} | Nettoyé: '{span_nettoye}'")

Longueur du texte : 2783 caractères

Nombre total de détections : 8

❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
✅ PROTÉGÉ | Span: 'Hurler' | Score: 0.85 | Nettoyé: 'hurler'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'
❌ SERA MASQUÉ | Span: 'MPS' | Score: 0.85 | Nettoyé: 'mps'


In [9]:
# Ajout de MPS et autres sigles médicaux courts problématiques

TERMES_MEDICAUX_PROTEGES.update({
    "mps",  # mucopolysaccharidose
    "ecg", "eeg", "irm", "tdm", "ete", "eto",  # sigles d'examens FR
    "ct", "mri", "ekg",  # sigles d'examens EN
})

print(f"Liste mise à jour : {len(TERMES_MEDICAUX_PROTEGES)} termes")

# Re-test sur le texte complet
resultats = analyzer.analyze(text=texte_complet, language="en", entities=ENTITES_CIBLEES)
resultats_filtres = [r for r in resultats if r.score >= SEUIL_CONFIANCE and not contient_terme_protege(texte_complet[r.start:r.end].lower().strip(" .,"))]

print(f"\nDétections avant filtre : {len(resultats)}")
print(f"Détections après filtre (à masquer) : {len(resultats_filtres)}")

anonymise_test = anonymizer.anonymize(text=texte_complet, analyzer_results=resultats_filtres)
print("\nAperçu du texte anonymisé :")
print(anonymise_test.text[:500])

Liste mise à jour : 73 termes

Détections avant filtre : 8
Détections après filtre (à masquer) : 0

Aperçu du texte anonymisé :
Mucopolysaccharidosis type I (MPS I), also known as Hurler syndrome, of Hurler-Scheie syndrome, is a rare genetic disorder caused by a deficiency of the enzyme alpha-L-iduronidase. This deficiency leads to the accumulation of glycosaminoglycans (GAGs) in cells, tissues, and organs, causing a range of symptoms and complications. The treatments for MPS I aim to manage the symptoms, slow disease progression, and improve the quality of life of affected individuals. Here are some of the treatments:




In [10]:
# CELLULE 24quinquies (LOCAL) - Anonymisation finale définitive sur tout le dataset

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

df_sft_split = pd.read_csv("data_processed/dataset_sft_split.csv")

print(f"Anonymisation finale définitive de {len(df_sft_split)} lignes...\n")

df_sft_final_v4 = df_sft_split.copy()

print("Colonne 1/3 : instruction")
df_sft_final_v4["instruction"] = df_sft_final_v4.progress_apply(
    lambda row: anonymiser_texte_v5(row["instruction"], row["langue"]), axis=1
)

print("\nColonne 2/3 : contexte_patient")
df_sft_final_v4["contexte_patient"] = df_sft_final_v4.progress_apply(
    lambda row: anonymiser_texte_v5(row["contexte_patient"], row["langue"]), axis=1
)

print("\nColonne 3/3 : reponse")
df_sft_final_v4["reponse"] = df_sft_final_v4.progress_apply(
    lambda row: anonymiser_texte_v5(row["reponse"], row["langue"]), axis=1
)

df_sft_final_v4.to_csv("data_processed/dataset_sft_anonymise_final_v4.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v4.csv")

# Vérification finale sur le cas MPS/Hurler
verif = df_sft_final_v4[df_sft_final_v4["reponse"].str.contains("MPS I", na=False)]
print(f"\nLignes avec 'MPS I' intact : {len(verif)}")
if len(verif) > 0:
    print(verif.iloc[0]["reponse"][:200])

Anonymisation finale définitive de 6534 lignes...

Colonne 1/3 : instruction


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 2/3 : contexte_patient


  0%|          | 0/6534 [00:00<?, ?it/s]


Colonne 3/3 : reponse


  0%|          | 0/6534 [00:00<?, ?it/s]


💾 Sauvegardé -> data_processed/dataset_sft_anonymise_final_v4.csv

Lignes avec 'MPS I' intact : 2
Mucopolysaccharidosis type I (MPS I), also known as Hurler syndrome, of Hurler-Scheie syndrome, is a rare genetic disorder caused by a deficiency of the enzyme alpha-L-iduronidase. This deficiency lea


In [1]:
# CELLULE 25bis (LOCAL) - Ré-upload de la version finale corrigée sur Hugging Face

from huggingface_hub import login
import getpass
from datasets import Dataset, DatasetDict
import pandas as pd

# Ré-authentification si besoin (le kernel local n'a peut-être pas gardé la session)
token = getpass.getpass("Colle ton token Hugging Face (write) : ")
login(token=token)

df_final = pd.read_csv("data_processed/dataset_sft_anonymise_final_v4.csv")

train_ds = Dataset.from_pandas(df_final[df_final["split"] == "train"].drop(columns=["split"]).reset_index(drop=True))
val_ds = Dataset.from_pandas(df_final[df_final["split"] == "val"].drop(columns=["split"]).reset_index(drop=True))
test_ds = Dataset.from_pandas(df_final[df_final["split"] == "test"].drop(columns=["split"]).reset_index(drop=True))

dataset_dict = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})

repo_id = "UserMarrakech/chsa-triage-medical-sft"
dataset_dict.push_to_hub(repo_id, private=True)

print(f"\n✅ Dataset final corrigé ré-uploadé sur https://huggingface.co/datasets/{repo_id}")

Colle ton token Hugging Face (write) :  ········


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Dataset final corrigé ré-uploadé sur https://huggingface.co/datasets/UserMarrakech/chsa-triage-medical-sft
